# E18 — IMU-anchored residual fusion: can audio be added to IMU without negative transfer?

**Context.** The setup:
- TII UAV Realistic Fault Dataset; 5-class propeller-fault severity.
- 95 missions; leave-one-day-out over 5 days (2021-12-22 has only 10 missions).

**Established facts:**
- IMU is the strongest modality: day-out window macro-F1 0.8347, against 0.807 for audio R0.
- HTT/LACA, cross-attention, routers and unconstrained fusion give no reliable gain.
- E16 (audio representations) and E19 (augmentation / CORAL / day-validation) were null.

**E18 tests IMU-ANCHORED RESIDUAL FUSION.** Audio may only add a *bounded* correction to the IMU logits:
z_final = z_IMU + α·Δz, with α ∈ [0, 1].

**Questions:**
1. Can audio information be added to IMU without negative transfer?
2. Does residual fusion beat simple weighted fusion?

**Pairing.**
- E18 reuses the project's existing pairing: v29's `paired_windows` (v24 CORRECTED_ALIGNMENT, copied verbatim). Each
  window takes its IMU samples (128 × 6, file order) **and** its audio (64 chunks × the 80 R0 features) from the
  **same 2.56 s span of the same mission**.
- Audio starts later than the IMU log (median offset ~19 s), so the paired windows cover only part of the IMU
  timeline. The coverage is printed (hook 4).
- IMU-only (B0) is retrained and evaluated on exactly these paired windows. The unpaired E19/reference IMU numbers
  are only a soft check.

| arm | what it is |
|---|---|
| **B0 IMU-ONLY** | IMUMamba, reference training, on paired windows. Its checkpoint is the frozen **anchor** for R1 / C1 / C2 |
| AUDIO-ONLY | AudioMamba (R0 features), reference training. Needed for B1; reported as a reference row |
| **B1 WEIGHTED** | p = w·p_IMU + (1 − w)·p_audio from the B0 and AUDIO checkpoints; w ∈ {0.5, …, 1.0} chosen on inner-validation windows only; no training |
| **B2 JOINT** | IMU and audio encoders trained end-to-end; [h_I; h_A] → linear head; CE only |
| **R1 RESIDUAL** | frozen B0 anchor + a new AudioMamba encoder; Δz = W2·relu(W1[h_I; h_A]) with W2 and its bias **zero-initialised**; α = 1.0·sigmoid(g[h_I; h_A]); loss = CE(z_final) + 0.01·mean‖αΔz‖² |
| **R2 RESIDUAL-FT** | from R1's state: the IMU model is unfrozen with lr × 0.1; loss = CE(z_final) + 0.5·CE(z_IMU) + 0.01·‖αΔz‖²; 10 extra epochs; best-val checkpoint over R1's end and R2's epochs |
| C1 R1-IMUONLY | control: R1 whose correction head sees only h_I (no audio). Tests extra capacity alone |
| C2 R1-SHUFFLED | control: R1 with the audio windows permuted within each split (seeded). Tests that a gain needs **aligned** audio |

The InfoNCE and KL losses of earlier fusion notebooks are **dropped** and not added back.

**Protocol (identical to E19):**
- Fold k tests on day k. Inner validation as in E19 (v27's rule if found, else the seed-2026 fallback).
- Train-only z-score for each modality; inverse-frequency class weights from the fitting windows.
- 20 epochs, AdamW (lr 1e-4, weight decay 1e-4), cosine to 1e-6, clip 1.0, batch 32, best-val-macro-F1 checkpoint.
- Real `mamba_ssm`.
- Mission-cluster bootstrap: 2000 resamples, seed 2028.
- **SEEDS = [42, 123, 2024, 7, 99].**

In [ ]:
# ==============================================================================
# E18 RUN FLAGS (the only switches a user needs)
# ==============================================================================
SMOKE = False          # True: fold 1, seed 42, 2 epochs (R2: 2 extra epochs), every arm -> *_smoke outputs, no analysis
RUN_SUBSET = None      # None = everything; or e.g. {"arms": ["B0", "R1"], "folds": [0, 1], "seeds": [42]}
                       # (missing dependencies are added automatically and printed: R2 needs R1; R1/C1/C2 need B0; B1 needs B0 + AUD)

## 0. Setup (E19 template: audio reference 0a–0d + IMU reference constants, verbatim except output names and hook names)

In [ ]:
# ==============================================================================
# 0a. ENVIRONMENT, PATHS, DATASET
# ==============================================================================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must be set before the first cuBLAS call (determinism)

import ast
import copy
import csv
import gc
import glob
import hashlib
import json
import math
import random
import re
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
if IS_KAGGLE:
    BASE, OUT = Path("/kaggle/working/tii"), Path("/kaggle/working")
elif IS_COLAB:
    BASE, OUT = Path("/content/tii"), Path("/content")
else:
    BASE, OUT = Path.cwd() / "tii", Path.cwd() / "e18_outputs"
BASE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
ENV = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "local")

REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p: Path) -> bool:
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_attached = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _attached += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _attached:
    DATA_ROOT, DATA_SOURCE = _attached[0], "attached Kaggle dataset"
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"Cloning {REPO_URL} (--depth 1) into {_repo} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT, DATA_SOURCE = _repo / "Dataset", "git clone --depth 1"
assert _is_dataset_root(DATA_ROOT), f"{DATA_ROOT} does not contain class folders 0..4"

FIG_DIR = OUT / "figures"
CACHE_DIR = OUT / "e18_audio_cache"          # E18: own cache dir
PROBS_DIR = OUT / "e18_probs"
CKPT_DIR = OUT / "e18_checkpoints"
RESULTS_CSV = OUT / "e18_results.csv"
for _d in (FIG_DIR, CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV} | BASE {BASE} | outputs {OUT}")
print(f"Dataset: {DATA_ROOT} ({DATA_SOURCE})")
print(f"Figures -> {FIG_DIR} | cache -> {CACHE_DIR} | results -> {RESULTS_CSV} | probs -> {PROBS_DIR}")

In [ ]:
# ==============================================================================
# 0b. REAL mamba-ssm (no fallback). Installs only if it is not importable already.
# ==============================================================================
import importlib
import importlib.util

import torch


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Real mamba-ssm needs a GPU: enable a GPU accelerator (Kaggle: Settings -> "
                       "Accelerator -> GPU) and re-run. There is deliberately no CPU fallback in this notebook.")

if importlib.util.find_spec("mamba_ssm") is None:
    print(f"Installing mamba-ssm (torch {torch.__version__}, CUDA {torch.version.cuda}, python "
          f"{sys.version_info.major}.{sys.version_info.minor}) ...")
    _ok = (_pip(["packaging", "ninja"])
           and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output above). Check that internet is ON and a GPU "
                           "is attached, then restart the session and re-run. No fallback model is used.")

try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"mamba_ssm could not be imported ({exc!r}). Restart the kernel after installation and re-run; "
                       "no fallback model is used.") from exc
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} imported: {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | "
      f"torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 0c. CONSTANTS — every fixed value lives here
# ==============================================================================
SEEDS = [42, 123, 2024]

# --- AudioBuffer layout: 1536 = 256 frames x 6 interleaved slots; sample i -> frame i // 6, slot i % 6 ---
N_SLOTS = 6
DROP_SLOT = 5                                   # 0-based slot index 5 (the 6th slot): identically zero, dropped
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
N_CH = len(KEEP_SLOTS)                          # 5 audio channels
FRAME = 256                                     # frames per buffer = samples per channel per buffer
BUFFER_LEN = N_SLOTS * FRAME                    # 1536
N_RFFT = FRAME // 2 + 1                         # 129 per-channel rFFT bins (0..128)
KEEP_BINS = list(range(1, 17))                  # model input: per-channel bins 1..16 (fixed a priori)
N_FEAT = N_CH * len(KEEP_BINS)                  # 80

# --- windows (chunk counts only: no IMU, no time axis) ---
WINDOW_CHUNKS = 128
STRIDE_CHUNKS = 64
TA = 64                                         # tokens per window
TOKEN_POOL = WINDOW_CHUNKS // TA                # 2 consecutive chunks averaged per token
STEADY = (0.10, 0.90)                           # steady flight = 10%..90% of a mission's chunks (by position)

ASSUMED_CHANNEL_RATE_HZ = 16000                 # UNVERIFIED, used ONLY for secondary axis labels, never in any computation

# --- dataset facts (asserted in section 1) ---
N_CLASSES = 5
EXCLUDE_RATIO = 0.5                             # exclude if audio_start_offset / mission_duration >= this
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}        # the single remaining 12-29 mission joins the 12-31 group
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"

# --- model + training: transcribed from version 28/minor-project-v28.ipynb (config cell, UnimodalMamba, train_unimodal),
#     the audio arms A1/A2. E19: used unchanged; the v28 notebook is not read. ---
D_MODEL = 256
DROPOUT = 0.15
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496                 # 2 x 437,248 (real mamba_ssm, incl. the pre-norm LayerNorms)
EPOCHS = 20                                     # fixed; no early stopping in v28
BATCH_SIZE = 32
LEARNING_RATE = 1e-4                            # AdamW
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6                                   # CosineAnnealingLR(T_max=EPOCHS, eta_min=LR_MIN), stepped per epoch
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2
# checkpoint rule (v28): after every epoch, keep the state with the highest validation macro-F1 (strictly greater)

# --- splits ---
INNER_VAL_FRACTION, INNER_VAL_SEED = 0.15, 2027                    # v27 _split_inner_val (day-out protocol index 0)
INNER_VAL_FALLBACK_SEED = 2026                                     # used ONLY if v27's notebook cannot be read
V27_INNER_VAL_SHA1 = "58e3c037c513ace2"         # sha1[:16] of v27's _split_inner_val source (ast segment)

# --- analysis (fixed before seeing results) ---
N_PERMUTATIONS = 1000
PERM_SEED = 2029
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
TOP_CELLS = 10
SIGN_CONSISTENT_MIN_DAYS = 4
MIN_PER_CLASS_FOR_DAY_SIGN = 2
SPEARMAN_THRESHOLD = 0.7
PCA_BINS = list(range(1, 33))
EVIDENCE_K = [1, 2, 4, 8, 16, 32]

CHECK_DETERMINISM = False                       # E19: no hard determinism gate (mamba-ssm CUDA is non-deterministic); A0 vs reference is a soft check
DETERMINISM_EPOCHS = 3

HZ_PER_BIN = ASSUMED_CHANNEL_RATE_HZ / FRAME    # 62.5 Hz/bin — secondary axis labels ONLY
print(f"{N_SLOTS} slots x {FRAME} frames = {BUFFER_LEN} samples per buffer | channels kept {[s + 1 for s in KEEP_SLOTS]} "
      f"(slot {DROP_SLOT + 1} dropped) | model input {N_CH} ch x bins {KEEP_BINS[0]}-{KEEP_BINS[-1]} = {N_FEAT} features")
print(f"Windows: {WINDOW_CHUNKS} chunks, stride {STRIDE_CHUNKS}, {TA} tokens of {TOKEN_POOL} chunks | seeds {SEEDS}")

In [ ]:
# ==============================================================================
# 0c-IMU. IMU reference constants (Ran_TII_IMU_Only_Analysis_Mamba.ipynb, cell 'CONSTANTS', verbatim lines) +
#         a check that every setting SHARED with the audio reference is identical
# ==============================================================================
CHANNELS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
WIN_S = 2.56                     # WINDOW_SEC
STRIDE_S = 1.28                  # STEP_SEC
TS = 128                         # TS_SENSOR: samples per window, chosen evenly BY INDEX among the window's file-order samples
MIN_SAMPLES_PER_WINDOW = 10      # MIN_SENSOR_SAMPLES_PER_WINDOW
SENSOR_STD_EPS = 1e-6

IMU_REFERENCE_SHARED = {'SEEDS': [42, 123, 2024], 'N_CLASSES': 5, 'EXPECTED_EXCLUDED': {'rosbag2_2021_12_31-10_04_22', 'rosbag2_2021_12_31-09_58_13', 'rosbag2_2021_12_29-09_10_31', 'rosbag2_2021_12_31-10_12_31'}, 'EXPECTED_N_MISSIONS': 95, 'EXPECTED_CLASS_COUNTS': [18, 19, 18, 20, 20], 'DAYS': ['2021-12-22', '2021-12-31', '2022-01-04', '2022-01-05', '2022-01-10'], 'DAY_MERGE': {'2021-12-29': '2021-12-31'}, 'EXPECTED_DAY_COUNTS': [10, 36, 17, 15, 17], 'SMALL_FOLD_DAY': '2021-12-22', 'STEADY': (0.1, 0.9), 'D_MODEL': 256, 'DROPOUT': 0.15, 'MAMBA_NUM_BLOCKS': 2, 'MAMBA_D_STATE': 16, 'MAMBA_D_CONV': 2, 'MAMBA_EXPAND': 2, 'EXPECTED_MAMBA_PARAMS': 874496, 'EPOCHS': 20, 'BATCH_SIZE': 32, 'LEARNING_RATE': 0.0001, 'WEIGHT_DECAY': 0.0001, 'LR_MIN': 1e-06, 'GRAD_CLIP_NORM': 1.0, 'NUM_WORKERS': 2, 'INNER_VAL_FRACTION': 0.15, 'INNER_VAL_SEED': 2027, 'INNER_VAL_FALLBACK_SEED': 2026, 'V27_INNER_VAL_SHA1': '58e3c037c513ace2', 'BOOTSTRAP_RESAMPLES': 2000, 'BOOTSTRAP_SEED': 2028}
_diff = {k: (globals()[k], v) for k, v in IMU_REFERENCE_SHARED.items() if globals()[k] != v}
assert not _diff, f"IMU and audio reference settings differ: {_diff}"
print(f"IMU constants: windows {WIN_S} s / stride {STRIDE_S} s, Ts {TS}, channels {CHANNELS}; all "
      f"{len(IMU_REFERENCE_SHARED)} settings shared with the audio reference are identical")


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def find_file(fname, subdirs):
    cands = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        cands += [str(base / s / fname) for s in subdirs] + [str(base / fname)]
    return next((Path(c) for c in cands if Path(c).is_file()), None)

In [ ]:
# ==============================================================================
# 0c-v29. PAIRED-WINDOW CONSTANTS (version 29/minor-project-v29.ipynb, cell 2c, verbatim lines) + equality checks
# ==============================================================================
N_AUDIO_FEAT = len(KEEP_SLOTS) * len(KEEP_BINS)  # 80
AUDIO_STD_EPS = 1e-6
WINDOW_SEC = 2.56
STEP_SEC = 1.28
TA_AUDIO = 64
TS_SENSOR = 128
MIN_SENSOR_SAMPLES_PER_WINDOW = 10
N_SENSOR_CHANNELS = 6
N_HASH_SAMPLE = 25

assert (WINDOW_SEC, STEP_SEC, TS_SENSOR, MIN_SENSOR_SAMPLES_PER_WINDOW, N_SENSOR_CHANNELS) == \
       (WIN_S, STRIDE_S, TS, MIN_SAMPLES_PER_WINDOW, len(CHANNELS)), "v29 IMU window constants differ from the IMU reference"
assert (N_AUDIO_FEAT, TA_AUDIO) == (N_FEAT, TA), "v29 audio constants differ from the audio reference (R0: 80 features, 64 tokens)"
print(f"Paired windows: {WINDOW_SEC} s, step {STEP_SEC} s | IMU {TS_SENSOR} x {N_SENSOR_CHANNELS} | audio {TA_AUDIO} x "
      f"{N_AUDIO_FEAT} (R0 features) — identical to the IMU and audio references")

In [ ]:
# ==============================================================================
# 0c-E18. E18 CONSTANTS (arms, residual design, rule, budget) — ALL fixed a priori, never tuned
# ==============================================================================
SEEDS = [42, 123, 2024, 7, 99]                     # E18: >= 5 seeds (the reference constants above list 3)
ARMS = ["B0", "AUD", "B1", "B2", "R1", "R2", "C1", "C2"]   # run order inside every (fold, seed)
CANDIDATES = ["B1", "B2", "R1", "R2"]              # K = 4, each vs B0
CONTROLS = ["C1", "C2"]
ARM_NAMES = {"B0": "IMU-ONLY (anchor)", "AUD": "AUDIO-ONLY (reference)", "B1": "WEIGHTED LATE FUSION",
             "B2": "JOINT FUSION", "R1": "RESIDUAL (frozen anchor)", "R2": "RESIDUAL (joint fine-tune)",
             "C1": "CONTROL R1-IMUONLY", "C2": "CONTROL R1-SHUFFLED"}
DEPENDS = {"B1": ["B0", "AUD"], "R1": ["B0"], "C1": ["B0"], "C2": ["B0"], "R2": ["R1"]}
CKPT_ARMS = {"B0", "AUD", "R1"}                    # always checkpointed: later arms load them

# ---- residual fusion (fixed; never tuned) ----
ALPHA_MAX = 1.0                                    # alpha = ALPHA_MAX * sigmoid(g(.)) in [0, ALPHA_MAX]
LAMBDA_R = 0.01                                    # weight of mean ||alpha * dz||^2
LAMBDA_I = 0.5                                     # R2 only: weight of CE(z_IMU)
R_HIDDEN = 128                                     # W1: Linear(d_in, 128) -> ReLU -> W2: Linear(128, 5) (zero-init)
R2_EPOCHS = 10
R2_IMU_LR_FACTOR = 0.1
B1_WEIGHTS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]        # ties on validation F1 -> the larger w (closer to IMU-only)
C2_PERM_SEED = 2031
ZERO_INIT_TOL = 1e-5
ANCHOR_REPRO_TOL = 1e-3                            # max |p_anchor - p_B0| on test windows (same weights, eval mode)
PAIRING_MIN_COVERAGE = 0.90

# ---- pre-registered rule ----
MIN_DELTA = 0.02
PAIRED_CI = 99.0
MIN_FOLDS_SAME_SIGN = 4
MIN_SEEDS_POSITIVE = 4
NO_HARM_C4_RECALL = 0.05
NO_HARM_FCN = 0.02
RULE_TEXT = 'PRE-REGISTERED DECISION RULE (E18; fixed before any E18 result; applied in code; adapted from E19)\nPrimary metric: pooled out-of-fold WINDOW macro-F1 from seed-averaged probabilities on the PAIRED test windows\n(95 missions, day-out, 5 seeds). Secondary: mission-level macro-F1, per-fold F1, worst-day F1, Class-4 recall,\n"faulty called Normal" rate. Every arm is evaluated on exactly the same paired test windows and missions.\nCandidates: B1 WEIGHTED, B2 JOINT, R1 RESIDUAL (frozen anchor), R2 RESIDUAL (joint fine-tune) (K = 4),\neach against B0 IMU-ONLY retrained on the same paired windows.\nAn arm is CREDITED only if ALL hold vs B0:\n  1. pooled paired delta window macro-F1 >= +0.02;\n  2. the 99% paired mission-cluster bootstrap CI of the delta excludes 0 (2000 resamples of missions within class,\n     seed 2028, the SAME resampled missions for both arms);\n  3. delta > 0 in >= 4 of 5 folds (seed-averaged, per held-out day);\n  4. delta > 0 in >= 4 of 5 seeds (each seed\'s own pooled out-of-fold F1);\n  5. worst-day F1 not below B0\'s (min over folds of the seed-averaged fold F1);\n  6. NO-HARM: Class-4 recall not more than 0.05 below B0\'s, and the "faulty called Normal" rate not more than 0.02\n     above B0\'s (pooled, seed-averaged, window level).\nWORSE (mirror): delta <= -0.02, the 99% CI entirely below 0, and delta < 0 in >= 4 of 5 folds.\nOtherwise: NOT CREDITED.\nConfirmatory contrast (not part of K): R1 and R2 vs B1 with conditions 1-5 -> "BEATS WEIGHTED FUSION" or\n"DOES NOT BEAT WEIGHTED FUSION". A residual arm credited vs B0 that does not beat B1 is reported as exactly that.\nControl read-out: if C1 (R1-IMUONLY) or C2 (R1-SHUFFLED) reaches conditions 1-3 vs B0, "GAIN NOT ATTRIBUTABLE TO AUDIO"\nis printed next to the R1/R2 verdicts. C1, C2 and AUDIO-ONLY are reported, never rated.\nEvery arm is reported whatever the outcome; a null result is a valid outcome.'

# ---- soft check vs E19 A0 (IMU) and the runtime budget ----
E19_A0_FALLBACK = [0.8324, 0.8489, 0.7260, 0.8316, 0.8129]   # IMU reference per-fold F1 if no e19_results.csv is found
B0_REF_TOL = 0.04
SEC_PER_RUN = {"B0": 62, "AUD": 57, "B1": 2, "B2": 120, "R1": 80, "R2": 60, "C1": 35, "C2": 80}   # Kaggle-T4 guesses
BUDGET_HOURS = 5.0

# ---- run plan ----
if SMOKE:
    FOLDS_TO_RUN, SEEDS_TO_RUN, EPOCHS_TO_RUN, R2_EPOCHS_TO_RUN = [1], [42], 2, 2
    RESULTS_CSV = OUT / "e18_results_smoke.csv"
    PROBS_DIR = OUT / "e18_probs_smoke"
    CKPT_DIR = OUT / "e18_checkpoints_smoke"
else:
    FOLDS_TO_RUN, SEEDS_TO_RUN, EPOCHS_TO_RUN, R2_EPOCHS_TO_RUN = list(range(len(DAYS))), SEEDS, EPOCHS, R2_EPOCHS
_sub = RUN_SUBSET or {}
FOLDS_TO_RUN = _sub.get("folds", FOLDS_TO_RUN)
SEEDS_TO_RUN = _sub.get("seeds", SEEDS_TO_RUN)
ARMS_TO_RUN = list(_sub.get("arms", ARMS))
_added = True
while _added:
    _added = False
    for a in list(ARMS_TO_RUN):
        for dep in DEPENDS.get(a, []):
            if dep not in ARMS_TO_RUN:
                ARMS_TO_RUN.append(dep)
                _added = True
                print(f"RUN_SUBSET: added dependency {dep} (needed by {a})")
ARMS_TO_RUN = [a for a in ARMS if a in ARMS_TO_RUN]
assert set(ARMS_TO_RUN) <= set(ARMS)
for _d in (PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
(OUT / ("e18_rule_smoke.txt" if SMOKE else "e18_rule.txt")).write_text(RULE_TEXT + "\n", encoding="utf-8")
print(f"SMOKE={SMOKE} | arms {ARMS_TO_RUN} | folds {FOLDS_TO_RUN} | seeds {SEEDS_TO_RUN} | epochs {EPOCHS_TO_RUN} "
      f"(R2 +{R2_EPOCHS_TO_RUN}) | results -> {RESULTS_CSV.name} | probs -> {PROBS_DIR.name}/ | rule -> e18_rule.txt")

In [ ]:
# ==============================================================================
# 0d. SEEDING / DETERMINISM, FILE-ACCESS LOGGER, PASS/FAIL HOOKS, FIGURE HELPERS
# ==============================================================================
def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed: int) -> torch.Generator:
    """DataLoader shuffling uses its own generator, so the shuffle order depends only on the seed."""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id: int) -> None:
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
set_all_seeds(SEEDS[0])
print("Determinism settings: cudnn.deterministic=True, cudnn.benchmark=False, "
      "torch.use_deterministic_algorithms(True, warn_only=True) (warn_only because mamba-ssm's custom CUDA kernels are "
      f"outside PyTorch's deterministic registry), CUBLAS_WORKSPACE_CONFIG={os.environ['CUBLAS_WORKSPACE_CONFIG']}, "
      "per-run seeding of random/numpy/torch/CUDA, a seeded torch.Generator for DataLoader shuffling, and worker_init_fn.")

# ---- dataset-file access logger: EVERY dataset file read in this notebook goes through open_logged ----
ACCESS_KINDS = ("AudioBuffer:first_record", "AudioBuffer:full_parse", "SensorCombined:first_last_timestamp")
FILE_LOG = Counter()
FILE_LOG_PER_MISSION = {k: Counter() for k in ACCESS_KINDS}


def open_logged(path, kind, mode="r"):
    assert kind in ACCESS_KINDS, f"unknown access kind {kind}"
    FILE_LOG[kind] += 1
    FILE_LOG_PER_MISSION[kind][Path(path).parent.name] += 1
    return open(path, mode) if "b" in mode else open(path, mode, encoding="utf-8")


# ---- verification hooks ----
HOOKS = defaultdict(list)
HOOK_NAMES = {1: "data: exclusion set, mission/class/day counts, slot 6 zero (v29 metadata)",
              2: "pairing: audio + IMU from the same span of one mission; hash rebuild; shapes (v29)",
              3: "split leakage: test day absent from fit/val; inner val has all 5 classes",
              4: "pairing coverage: >= 90% of paired windows hold real audio chunks (IMU-grid coverage printed)",
              5: "identical test window ids / missions / training pool across all arms per (fold, seed)",
              6: "real mamba_ssm + params", 7: "residual zero-init: z_final == z_IMU on the first batch before any update",
              8: "frozen anchor: IMU parameters unchanged by R1/C1/C2; anchor reproduces B0's test probabilities",
              9: "alpha within [0, ALPHA_MAX] on every evaluation pass",
              10: "C2 permutation stays inside its split, is not identity for > 95% of windows, labels untouched",
              11: "audio and IMU z-score statistics from training windows only",
              12: "B0 on paired windows vs E19 A0 per-fold F1 within 0.04 (SOFT)"}


def hook(n, ok, msg, stop=None):
    ok = bool(ok)
    HOOKS[n].append((ok, msg))
    print(f"[HOOK {n}] {'PASS' if ok else 'FAIL'} — {msg}")
    stop = (n <= 4) if stop is None else stop
    if not ok and stop:
        raise RuntimeError(f"HOOK {n} FAILED: {msg}. Stopping before anything is trained.")
    return ok


# ---- figures (palette: validated categorical order; sequential = one hue; diverging = two hues + neutral midpoint) ----
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CLASS_MARKERS = ["o", "s", "^", "D", "v"]
DAY_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
DAY_MARKERS = ["o", "s", "^", "D", "v"]
SEQ_CMAP, DIV_CMAP = "Blues", "RdBu_r"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "grid.linewidth": 0.6, "lines.linewidth": 1.5, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})
FREQ_LABEL = "per-channel FFT bin (1-128)"
HZ_LABEL = "approx. Hz if the per-channel rate were 16 kHz (unverified)"
FREQ_CAVEAT = ("Frequency axis = per-channel FFT bin of the de-interleaved channels. The Hz axis assumes a per-channel "
               "rate of 16 kHz, which is UNVERIFIED (the files carry no sample rate).")
FIGURES = []


def add_bin_axis(ax, axis="x"):
    """Secondary Hz axis for a per-channel-bin axis. Label only: HZ_PER_BIN is never used in a computation."""
    fwd, inv = (lambda b: b * HZ_PER_BIN), (lambda h: h / HZ_PER_BIN)
    if axis == "x":
        sec = ax.secondary_xaxis("top", functions=(fwd, inv))
        sec.set_xlabel(HZ_LABEL, fontsize=7)
    else:
        sec = ax.secondary_yaxis("right", functions=(fwd, inv))
        sec.set_ylabel(HZ_LABEL, fontsize=7)
    sec.tick_params(labelsize=7)
    ax._bin_caveat = True
    return sec


def save_fig(fig, name, caveat, freq_axes=()):
    """Every figure is saved as PNG with its caveat printed under it. Frequency figures must carry the Hz caveat."""
    assert caveat, f"{name}: every figure needs a caveat"
    freq_ok = all(getattr(ax, "_bin_caveat", False) for ax in freq_axes) and (not freq_axes or "UNVERIFIED" in caveat)
    fig.text(0.5, -0.01, caveat, ha="center", va="top", fontsize=8, color="#52514e", wrap=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    FIGURES.append({"name": name, "freq": bool(freq_axes), "freq_caveat_ok": freq_ok})
    print(f"  saved {FIG_DIR / (name + '.png')}" + ("" if freq_ok else "  <- MISSING frequency caveat"))


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

## 1. Data: parse once, then ONE set of paired windows (v29 cells 3a–3c, verbatim; hooks renumbered)
- **3a — parse.**
  - IMU: the IMU notebook's schema-first parser, **file order, never sorted**.
  - Audio: the audio notebook's per-chunk 80 R0 features (`pow[:, :, 1:17] ** 0.25`) plus `recording_started_time`.
- **3b — metadata.** v24's corrected audio anchor, then the exclusion rule computed from metadata (offset / duration
  ≥ 0.5 → excluded; it must equal the 4 known names), then the counts.
- **3c — paired windows.**
  - Audio (64 × 80) and IMU (128 × 6) come from the **same real time span of the same mission**. The window grid
    starts at audio onset with a 1.28 s step.
  - IMU: 128 real file-order samples chosen evenly by index.
  - Audio: 64 chunks chosen evenly by index among the chunks inside the span.
  - Includes a hash rebuild of 25 sampled windows.

In [ ]:
# ==============================================================================
# 3a. PARSE (all 99 folders: the 4 exclusions are computed from metadata, not assumed) -> v29_cache/<mission>.npz
# ==============================================================================
def unwrap_message(obj):                                   # IMU notebook, unchanged
    if isinstance(obj, dict):
        for key in ("msg", "message", "data"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def get_vec(d, candidates):                                # IMU notebook, unchanged
    for key in candidates:
        if key in d:
            v = d[key]
            if isinstance(v, (list, tuple)) and len(v) >= 3:
                return np.asarray(v[:3], dtype=np.float64)
        vals, ok = [], True
        for i in range(3):
            k = key.format(i=i)
            if k in d:
                vals.append(d[k])
            else:
                ok = False
                break
        if ok:
            return np.asarray(vals, dtype=np.float64)
    return None


def parse_imu_file(path):
    """IMU notebook's parser: rows [timestamp_us, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z] in FILE ORDER."""
    rows, bad = [], 0
    with open(path, "r") as f:
        for line in f:
            try:
                obj = unwrap_message(json.loads(line))
                ts = obj.get("timestamp", obj.get("time_stamp", None))
                acc = get_vec(obj, ["accelerometer_m_s2", "accelerometer_m_s2[{i}]", "accelerometer", "acceleration"])
                gyro = get_vec(obj, ["gyro_rad", "gyro_rad[{i}]", "gyro", "angular_velocity"])
                if ts is None or acc is None or gyro is None:
                    bad += 1
                    continue
                rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
            except Exception:
                bad += 1
    return np.asarray(rows, dtype=np.float64).reshape(-1, 7), bad


def parse_mission(mdir):
    cache = CACHE_DIR / f"{mdir.name}.npz"
    if cache.exists():
        z = np.load(cache)
        return {k: z[k] for k in z.files}
    s_path = sorted(mdir.glob("*SensorCombined*.jsonl"))[0]
    a_path = sorted(mdir.glob("*AudioBuffer*.jsonl"))[0]
    imu, bad = parse_imu_file(s_path)
    chunks, lens, rst, rsts = [], Counter(), None, set()
    with open(a_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            x = rec.get("data")
            if x is None or len(x) == 0:
                continue
            if "recording_started_time" in rec:
                rsts.add(float(rec["recording_started_time"]))
                if rst is None:
                    rst = float(rec["recording_started_time"])          # v24: the FIRST record's value
            lens[len(x)] += 1
            chunks.append(x)
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal}"
    assert rst is not None, f"{mdir.name}: no recording_started_time — the audio cannot be anchored"
    keep_idx = np.array([i for i, x in enumerate(chunks) if len(x) == modal], dtype=np.int64)
    raw = np.asarray([chunks[i] for i in keep_idx], dtype=np.int64)
    n_all = len(chunks)
    del chunks
    frames = raw.reshape(len(raw), FRAME, N_SLOTS)                # sample i -> frame i // 6, slot i % 6
    slot6_zero = bool(np.all(frames[:, :, DROP_SLOT] == 0))
    ch = frames[:, :, KEEP_SLOTS].astype(np.float64)
    ch = ch - ch.mean(axis=1, keepdims=True)                      # per-chunk mean removed, per channel
    pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)    # (n, 5, 129)
    feats = (pw[:, :, KEEP_BINS] ** 0.25).reshape(len(pw), N_AUDIO_FEAT).astype(np.float32)   # sqrt|rFFT|, channel-major
    out = {"imu_t": imu[:, 0], "imu_x": imu[:, 1:].astype(np.float32), "imu_bad": bad, "feats": feats,
           "keep_idx": keep_idx, "n_all_chunks": n_all, "rst": rst, "n_rst_values": len(rsts), "slot6_zero": slot6_zero}
    tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, cache)
    return out


def mission_day(name):
    d = name[8:18].replace("_", "-")
    return DAY_MERGE.get(d, d)


ALL_DIRS, LABEL_OF = [], {}
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not (class_dir.is_dir() and class_dir.name.isdigit()):
        continue
    for mdir in sorted(class_dir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            ALL_DIRS.append(mdir)
            LABEL_OF[mdir.name] = int(class_dir.name)
_t0 = time.time()
PARSED = {}
for i, mdir in enumerate(ALL_DIRS, 1):
    PARSED[mdir.name] = parse_mission(mdir)
    if i % 10 == 0 or i == len(ALL_DIRS):
        print(f"  parsed {i}/{len(ALL_DIRS)} missions | {time.time() - _t0:.0f} s")

In [ ]:
# ==============================================================================
# 3b. METADATA — v24's corrected audio anchor, exclusion rule, mission / class / day counts (HOOK data)
# ==============================================================================
def timebase(P):
    """v24 CORRECTED_ALIGNMENT time axes. IMU: FILE ORDER, seconds from the first sample. Audio: anchored at
    recording_started_time; chunk duration = (mission duration - offset) / number of chunks."""
    t_us = P["imu_t"]
    s_times = (t_us - t_us[0]) / 1e6
    duration = float(s_times[-1] - s_times[0])
    offset = (float(P["rst"]) - float(t_us[0])) / 1e6
    chunk = (duration - offset) / int(P["n_all_chunks"])
    a_times = (offset + np.arange(int(P["n_all_chunks"]), dtype=np.float64) * chunk)[P["keep_idx"]]
    return s_times, a_times, duration, offset, chunk


META = []
for name, P in PARSED.items():
    s_times, a_times, dur, off, ch = timebase(P)
    META.append({"mission": name, "class": LABEL_OF[name], "day": mission_day(name), "duration_s": dur, "offset_s": off,
                 "ratio": off / dur, "chunk_ms": 1e3 * ch, "n_imu": len(s_times), "imu_inversions": int((np.diff(s_times) < 0).sum()),
                 "n_chunks": int(P["n_all_chunks"]), "chunks_dropped": int(P["n_all_chunks"]) - len(P["keep_idx"]),
                 "rst_values": int(P["n_rst_values"]), "slot6_zero": bool(P["slot6_zero"])})
META = pd.DataFrame(META).set_index("mission")
EXCLUDED = set(META.index[META.ratio >= EXCLUDE_RATIO])
KEPT = sorted(set(META.index) - EXCLUDED)
M = META.loc[KEPT]
LABEL = {n: int(M.loc[n, "class"]) for n in KEPT}
DAY = {n: M.loc[n, "day"] for n in KEPT}
_cls = np.bincount(M["class"], minlength=N_CLASSES).tolist()
_days = [int((M.day == d).sum()) for d in DAYS]
print(META.loc[sorted(EXCLUDED), ["class", "day", "offset_s", "duration_s", "ratio"]].round(3).to_string())
print(f"\nKept {len(KEPT)} missions | max kept ratio {M.ratio.max():.3f} | median offset {M.offset_s.median():.2f} s | "
      f"chunk duration {M.chunk_ms.median():.2f} ms (median; v24's corrected-anchor formula) | "
      f"non-modal chunks dropped {int(M.chunks_dropped.sum())} | missions with IMU timestamp inversions "
      f"{int((M.imu_inversions > 0).sum())} (total {int(M.imu_inversions.sum())})")
print(pd.crosstab(M.day, M["class"]).reindex(DAYS).to_string())
hook(1, EXCLUDED == EXPECTED_EXCLUDED, f"exclusion set == the 4 known names: {sorted(EXCLUDED)}")
hook(1, len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and _days == EXPECTED_DAY_COUNTS,
     f"{len(KEPT)} missions, classes {_cls}, days {_days} (expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, {EXPECTED_DAY_COUNTS})")
hook(1, bool(M.slot6_zero.all()) and bool((M.rst_values == 1).all()),
     f"audio slot 6 identically zero in {int(M.slot6_zero.sum())}/{len(M)}; recording_started_time constant in "
     f"{int((M.rst_values == 1).sum())}/{len(M)}")

In [ ]:
# ==============================================================================
# 3c. ONE SET OF PAIRED WINDOWS (v24 CORRECTED_ALIGNMENT): audio (64 x 80) and IMU (128 x 6, file order) from the SAME
#     real time span of the SAME mission. Every model below reads these arrays (unimodal models ignore one half).
# ==============================================================================
def paired_windows(name, P):
    s_times, a_times, dur, _, _ = timebase(P)
    tel, feats = P["imu_x"], P["feats"]
    max_time = min(s_times[-1], a_times[-1])
    out, i = [], 0
    t_start = max(0.0, float(a_times[0]))                  # v24: CORRECTED_WINDOW_START = "audio_onset"
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue
        s_idx = np.where(s_mask)[0]
        pick = s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]
        tau_s_abs = s_times[pick]
        x_s = tel[pick].astype(np.float32)                 # real file-order samples (IMU notebook rule)
        x_s_v24 = np.stack([np.interp(tau_s_abs, s_times[s_mask], tel[s_mask, c]) for c in range(N_SENSOR_CHANNELS)],
                           axis=1).astype(np.float32)      # v24's np.interp form, kept only for the comparison count
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            a_pick = a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]
            tau_a_abs = a_times[a_pick]
            x_a = feats[a_pick].astype(np.float32)         # == v24's np.interp at exact (strictly increasing) chunk times
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
            x_a = np.stack([np.interp(tau_a_abs, a_times, feats[:, c], left=0.0, right=0.0) for c in range(N_AUDIO_FEAT)],
                           axis=1).astype(np.float32)
        rel = (t_start + WINDOW_SEC / 2) / dur
        out.append({"window_id": f"{name}#{i:04d}", "mission": name, "cls": LABEL[name], "day": DAY[name],
                    "t_start": t_start, "t_end": t_end,
                    "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady"),
                    "x_a": x_a, "x_s": x_s, "tau_a": ((tau_a_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "tau_s": ((tau_s_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "v24_interp_differs": bool(not np.array_equal(x_s, x_s_v24)),
                    "v24_interp_maxdiff": float(np.abs(x_s - x_s_v24).max())})
        i += 1
        t_start += STEP_SEC                                 # v24: accumulated float step
    return out


_W = []
for name in KEPT:
    _W.extend(paired_windows(name, PARSED[name]))
WMETA = pd.DataFrame([{k: v for k, v in w.items() if k not in ("x_a", "x_s", "tau_a", "tau_s")} for w in _W])
XA = np.stack([w["x_a"] for w in _W])                      # (N, 64, 80)
XS = np.stack([w["x_s"] for w in _W])                      # (N, 128, 6)
TAU_A = np.stack([w["tau_a"] for w in _W])                 # (N, 64)   in window units (v24 dataset: tau / window_sec)
TAU_S = np.stack([w["tau_s"] for w in _W])                 # (N, 128)
YW = WMETA.cls.values.astype(np.int64)
del _W
gc.collect()
print(f"Paired windows: {len(WMETA)} | XA {XA.shape} | XS {XS.shape} | per class {np.bincount(YW, minlength=N_CLASSES).tolist()} | "
      f"per phase {WMETA.phase.value_counts().to_dict()}")
print(f"IMU: windows where the file-order direct pick differs from v24's np.interp form: "
      f"{int(WMETA.v24_interp_differs.sum())} / {len(WMETA)} (max abs diff {WMETA.v24_interp_maxdiff.max():.4g}) — "
      "nonzero only where a window's IMU timestamps go backwards")

# ---- pairing hooks ----
_span_ok = bool((TAU_A >= -1e-6).all() and (TAU_A <= 1 + 1e-6).all() and (TAU_S >= -1e-6).all() and (TAU_S <= 1 + 1e-6).all())
hook(2, _span_ok and WMETA.window_id.is_unique,
     "every window's audio tokens AND IMU samples lie inside its own [t_start, t_start + 2.56 s] span of ONE mission "
     "(both come from a single record: same mission id, same span)")
_rng_h = np.random.default_rng(0)
_sample = np.sort(_rng_h.choice(len(WMETA), size=min(N_HASH_SAMPLE, len(WMETA)), replace=False))
_hash_ok = True
for j in _sample:
    r = WMETA.iloc[j]
    rebuilt = {w["window_id"]: w for w in paired_windows(r.mission, PARSED[r.mission])}[r.window_id]
    same = sha16(rebuilt["x_a"], rebuilt["x_s"]) == sha16(XA[j], XS[j]) and abs(rebuilt["t_start"] - r.t_start) < 1e-12
    _hash_ok &= same
hook(2, _hash_ok, f"hash check on {len(_sample)} sampled windows: stored (audio, IMU) == rebuilt from the raw "
                          f"mission at the recorded (mission, t_start)")
_fin = bool(np.isfinite(XA).all() and np.isfinite(XS).all())
hook(2, _fin and XA.shape[1:] == (TA_AUDIO, N_AUDIO_FEAT) and XS.shape[1:] == (TS_SENSOR, N_SENSOR_CHANNELS),
     f"shapes audio {XA.shape[1:]}, IMU {XS.shape[1:]}; all finite: {_fin}")
PAIRED_SHA = sha16(XA, XS)
print(f"Paired-window content sha1: {PAIRED_SHA}")

In [ ]:
# ==============================================================================
# 1d. PAIRING COVERAGE (HOOK 4) + compatibility names for the E19 fold cell
# ==============================================================================
_rows = []
for name in KEPT:
    s_times, a_times, dur, off, ch = timebase(PARSED[name])
    w = WMETA[WMETA.mission == name]
    n_real = np.array([int(((a_times >= a) & (a_times <= b)).sum()) for a, b in zip(w.t_start, w.t_end)])
    # IMU reference grid (IMU notebook: 2.56 s windows, 1.28 s stride from the first file-order sample)
    imu_starts = np.arange(0, s_times[-1] - WIN_S + 1e-9, STRIDE_S)
    covered = np.array([bool(len(w) and np.min(np.abs(w.t_start.values - s)) <= STEP_SEC / 2 + 1e-9) for s in imu_starts])
    _rows.append({"mission": name, "paired_windows": len(w), "with_real_audio": int((n_real >= 2).sum()),
                  "median_chunks_per_window": float(np.median(n_real)) if len(n_real) else 0.0,
                  "imu_grid_windows": len(imu_starts), "imu_grid_covered": int(covered.sum()), "audio_offset_s": off})
PAIR_COV = pd.DataFrame(_rows).set_index("mission")
PAIR_COV.to_csv(OUT / "e18_pairing_coverage.csv")
REAL_AUDIO_SHARE = PAIR_COV.with_real_audio.sum() / PAIR_COV.paired_windows.sum()
IMU_GRID_COVERAGE = PAIR_COV.imu_grid_covered.sum() / PAIR_COV.imu_grid_windows.sum()
print(f"Paired windows {int(PAIR_COV.paired_windows.sum())} | with >= 2 real audio chunks: {100 * REAL_AUDIO_SHARE:.2f}% | "
      f"median audio chunks per window {PAIR_COV.median_chunks_per_window.median():.0f} (64 are picked by index)")
print(f"IMU-reference grid windows {int(PAIR_COV.imu_grid_windows.sum())} | covered by a paired window (>= 75% overlap): "
      f"{100 * IMU_GRID_COVERAGE:.1f}% | median audio offset {PAIR_COV.audio_offset_s.median():.1f} s (the IMU-only "
      f"stretch before audio onset has no audio and is NOT in any E18 arm)")
hook(4, REAL_AUDIO_SHARE >= PAIRING_MIN_COVERAGE,
     f"{100 * REAL_AUDIO_SHARE:.2f}% of paired windows carry real audio chunks (>= {100 * PAIRING_MIN_COVERAGE:.0f}% "
     f"required; the rest would be v29's zero-filled interpolation). Pairing method: v29 paired_windows "
     f"(v24 CORRECTED_ALIGNMENT, same span of the same mission).")
if IMU_GRID_COVERAGE < PAIRING_MIN_COVERAGE:
    print(f"WARN (soft): only {100 * IMU_GRID_COVERAGE:.1f}% of the IMU-only reference windows have a paired counterpart: "
          "audio starts after the IMU log. E18 compares arms on the paired windows only (B0 is retrained there), so the "
          "comparison stays paired; the absolute B0 score is not the unpaired IMU reference's.")

# ---- names used by the E19 fold cell (verbatim below) ----
missions = pd.DataFrame({"mission": KEPT, "class": [LABEL[n] for n in KEPT], "day": [DAY[n] for n in KEPT]})
imu_data = {n: {"day": DAY[n], "class": LABEL[n]} for n in KEPT}
groups = WMETA.mission.values
MISSION_DAY = {n: DAY[n] for n in KEPT}
MISSION_CLASS = {n: LABEL[n] for n in KEPT}

## 2. Day-out folds (E19 template = IMU reference fold cell, verbatim) and split hook 3
- Fold k tests on day k.
- Inner validation = v27's `_split_inner_val` if `minor-project-v27.ipynb` is attached under `/kaggle/input`;
  otherwise the fixed seed-2026 fallback. **The rule that ran is printed.**

In [ ]:
# ==============================================================================
# DAY-OUT FOLDS — identical to the audio notebook (fold k = DAYS[k]; v27 inner-validation rule, else seed 2026)
# ==============================================================================
def extract_function(nb_path,fname):
    try:
        nb=json.loads(Path(nb_path).read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"  could not read {nb_path}: {exc!r}")
        return None
    for cell in nb.get("cells",[]):
        if cell.get("cell_type")!="code":
            continue
        src=cell["source"] if isinstance(cell["source"],str) else "".join(cell["source"])
        src="\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!","%")))
        try:
            tree=ast.parse(src)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node,ast.FunctionDef) and node.name==fname:
                return ast.get_source_segment(src,node)
    return None


LABEL={Path(m):int(c) for m,c in zip(missions["mission"],missions["class"])}   # v27's rule keys missions by Path (.name)
_p27=find_file("minor-project-v27.ipynb",["version 27","Version 27"])
_src27=extract_function(_p27,"_split_inner_val") if _p27 else None
if _src27 is not None and hashlib.sha1(_src27.encode()).hexdigest()[:16]==V27_INNER_VAL_SHA1:
    _ns27={"np":np,"N_CLASSES":N_CLASSES,"LABEL":LABEL,"INNER_VAL_SEED":INNER_VAL_SEED,"INNER_VAL_FRACTION":INNER_VAL_FRACTION}
    exec(_src27,_ns27)
    inner_val=lambda pool,k: _ns27["_split_inner_val"](pool,0,k)
    INNER_VAL_RULE=f"v27 _split_inner_val (from {_p27}; seed [{INNER_VAL_SEED}, 0, fold]) — same as the audio notebook"
else:
    loud(("v27 notebook not found" if _p27 is None else f"v27 _split_inner_val unreadable/fingerprint mismatch at {_p27}")
         + f"\nFALLBACK inner validation: per-class ~15% (min 1) mission hold-out, fixed seed {INNER_VAL_FALLBACK_SEED} "
           "(the audio notebook's fallback, identical code).")

    def inner_val(pool,k):
        rng=np.random.default_rng([INNER_VAL_FALLBACK_SEED,k])
        train,val=[],[]
        for c in range(N_CLASSES):
            ms=sorted((m for m in pool if LABEL[m]==c),key=lambda m:m.name)
            perm=rng.permutation(len(ms))
            n_val=max(1,int(round(INNER_VAL_FRACTION*len(ms))))
            val+=[ms[i] for i in perm[:n_val]]
            train+=[ms[i] for i in perm[n_val:]]
        return train,val

    INNER_VAL_RULE=f"FALLBACK per-class hold-out, seed {INNER_VAL_FALLBACK_SEED} (same as the audio notebook's fallback)"
print("Inner-validation rule:",INNER_VAL_RULE)

FOLDS={}
_all=set(missions["mission"])
for k,day in enumerate(DAYS):
    test=sorted(missions.loc[missions["day"]==day,"mission"])
    tr,va=inner_val([Path(m) for m in sorted(_all-set(test))],k)
    FOLDS[k]={"day":day,"train":sorted(p.name for p in tr),"val":sorted(p.name for p in va),"test":test}
    f=FOLDS[k]
    assert not (set(f["train"])&set(f["val"])) and not (set(f["train"]+f["val"])&set(test))
    assert set(f["train"]+f["val"]+test)==_all
    assert day not in {imu_data[m]["day"] for m in f["train"]+f["val"]}, f"fold {k}: test day in training"
assert sorted(m for f in FOLDS.values() for m in f["test"])==sorted(_all)

print(f"{'fold':<5}{'test day':<12}{'missions tr/val/test':<22}{'test missions per class':<26}{'windows tr/val/test'}")
for k,f in FOLDS.items():
    per_class=np.bincount([imu_data[m]["class"] for m in f["test"]],minlength=N_CLASSES).tolist()
    n_w="/".join(str(int(np.isin(groups,f[s]).sum())) for s in ("train","val","test"))
    print(f"{k:<5}{f['day']:<12}{'/'.join(str(len(f[s])) for s in ('train','val','test')):<22}{str(per_class):<26}{n_w}"
          + ("   <- SMALL fold (10 missions)" if f["day"]==SMALL_FOLD_DAY else ""))

In [ ]:
# ==============================================================================
# 3b. SPLIT HOOK 3: no test-day mission in train/val; inner validation has all 5 classes; per-fold mission hashes
# ==============================================================================
_ok3 = True
for k, f in FOLDS.items():
    _ok3 &= f["day"] not in {MISSION_DAY[m] for m in f["train"] + f["val"]}
    _ok3 &= not (set(f["train"] + f["val"]) & set(f["test"])) and not (set(f["train"]) & set(f["val"]))
    _ok3 &= {MISSION_CLASS[m] for m in f["val"]} == set(range(N_CLASSES))
    _ok3 &= {MISSION_CLASS[m] for m in f["train"]} == set(range(N_CLASSES))
FOLD_HASH = {k: hashlib.sha1(json.dumps({s: f[s] for s in ("train", "val", "test")}).encode()).hexdigest()[:12]
             for k, f in FOLDS.items()}
hook(3, _ok3, "every fold: test day absent from train/val, no mission in two sets, inner validation and training "
              f"contain all 5 classes | rule: {INNER_VAL_RULE} | fold hashes {FOLD_HASH}")

## 3. Models (E19 template: AudioMamba + IMUMamba, references verbatim; real mamba_ssm)
The models are unchanged. Fusion arms read each model's pooled embedding (the input of its `classifier`) through a
forward pre-hook.

In [ ]:
# ==============================================================================
# 6a. MODELS: AudioMamba (audio reference) + IMUMamba (IMU reference) — real mamba_ssm (HOOK 6)
# ==============================================================================
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda")

# E19: the v28 notebook is not read; the reference's transcribed settings are used unchanged.


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM (v28)."""

    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(D_MODEL)
        self.mamba = MAMBA_CLASS(d_model=D_MODEL, d_state=MAMBA_D_STATE, d_conv=MAMBA_D_CONV, expand=MAMBA_EXPAND)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


class AudioMamba(nn.Module):
    """v28 UnimodalMamba, audio path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention
    pooling -> classifier."""

    def __init__(self, in_dim=N_FEAT):
        super().__init__()
        self.audio_encoder = nn.Sequential(nn.Linear(in_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, x):
        h = self.token_norm(self.audio_encoder(x) + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1))


def verify_model(model):
    real = all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks) and MAMBA_CLASS.__module__.startswith("mamba_ssm")
    blocks_ok = len(model.mamba_blocks) == MAMBA_NUM_BLOCKS and all(
        (b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND) for b in model.mamba_blocks)
    n_mamba = sum(p.numel() for p in model.mamba_blocks.parameters())
    n_params = sum(p.numel() for p in model.parameters())
    return real and blocks_ok and n_mamba == EXPECTED_MAMBA_PARAMS, n_params, n_mamba


set_all_seeds(SEEDS[0])
_probe = AudioMamba().to(device)
_ok, N_PARAMS, _n_mamba = verify_model(_probe)
assert _ok, "model is not the expected real-mamba_ssm architecture"
print(f"AudioMamba: {N_PARAMS:,} parameters (Mamba blocks {_n_mamba:,} == {EXPECTED_MAMBA_PARAMS:,}) | block class "
      f"{MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | modules {[n for n, _ in _probe.named_children()]}")
with torch.no_grad():
    assert _probe(torch.zeros(2, TA, N_FEAT, device=device)).shape == (2, N_CLASSES)
hook(6, _ok, f"every block is mamba_ssm.Mamba (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); {N_PARAMS:,} parameters",
     stop=False)
del _probe
torch.cuda.empty_cache()


# ---- IMUMamba: IMU reference, verbatim (it uses the same MambaBlock wrapper around mamba_ssm.Mamba) ----
class IMUMamba(nn.Module):
    """v28 UnimodalMamba, sensor path (S1)."""
    def __init__(self, in_dim=6, n_classes=N_CLASSES):
        super().__init__()
        self.sensor_encoder=nn.Sequential(
            nn.Conv1d(in_dim,64,kernel_size=5,padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64,D_MODEL,kernel_size=3,padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.modality_embedding=nn.Embedding(1,D_MODEL)
        self.token_norm=nn.LayerNorm(D_MODEL)
        self.mamba_blocks=nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn=nn.Sequential(nn.Linear(D_MODEL,64),nn.Tanh(),nn.Linear(64,1))
        self.classifier=nn.Sequential(nn.Dropout(DROPOUT),nn.Linear(D_MODEL,D_MODEL//2),nn.GELU(),
                                      nn.Linear(D_MODEL//2,n_classes))

    def forward(self,x):
        z=self.sensor_encoder(x.transpose(1,2)).transpose(1,2)
        h=self.token_norm(z+self.modality_embedding.weight.view(1,1,-1))
        for b in self.mamba_blocks:
            h=b(h)
        a=F.softmax(self.pool_attn(h),dim=1)
        return self.classifier((h*a).sum(dim=1))


DEVICE = device                                                  # the IMU reference's name for the same device
set_all_seeds(SEEDS[0])
_probe = IMUMamba().to(DEVICE)
_ok_i, N_PARAMS_IMU, _n_mamba_i = verify_model(_probe)
with torch.no_grad():
    _shape_i = _probe.eval()(torch.zeros(2, TS, len(CHANNELS), device=DEVICE)).shape == (2, N_CLASSES)
hook(6, _ok_i and _shape_i, f"IMUMamba: every block is mamba_ssm.Mamba (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); "
                            f"{N_PARAMS_IMU:,} parameters (Mamba {_n_mamba_i:,})", stop=True)
N_PARAMS_AUDIO = N_PARAMS
del _probe
torch.cuda.empty_cache()

## 4. E18 training core
- **One training loop for every arm**, which is the reference procedure:
  - `set_all_seeds(seed)` first;
  - AdamW (lr 1e-4, weight decay 1e-4), cosine to 1e-6 over the arm's epochs (stepped per epoch), grad clip 1.0,
    batch 32;
  - a seeded shuffle;
  - class-weighted CE with weights from the fitting windows;
  - validation macro-F1 after every epoch; checkpoint = the strictly best epoch.
- **z-score:** per fold, each modality is standardised with statistics from the **training windows only**. This is
  v29's FOLD_PREP rule: 80 audio features and 6 IMU channels.
- **Residual model:** `z_final = z_IMU + α·Δz` and `Δz = W2·relu(W1·h)`, with W2 and its bias set to **0**, so
  Δz ≡ 0 at step 0.
  - `α = ALPHA_MAX · sigmoid(g(h))`.
  - h = [h_I; h_A] for R1/R2/C2, and h_I for C1.
  - h_I is the frozen anchor's pooled embedding.
  - The anchor stays in eval mode with gradients off for R1/C1/C2. Its state hash is checked before and after
    training.
- **C2:** the audio windows are permuted **within** each split (fit / val / test), seeded per (fold, seed). Labels
  and IMU inputs are untouched.
- **R2:** starts from R1's best state. The anchor is unfrozen (train mode, lr × 0.1), with its own cosine over
  10 epochs. The incumbent checkpoint is R1's state (`selected_epoch = 0`) until an R2 epoch beats its validation F1.

In [ ]:
# ==============================================================================
# 4a. E18 TRAINING CORE (new code; B0/AUD == the reference training procedure)
# ==============================================================================
# mission aggregation: audio reference cell 6b, verbatim
def mission_level(missions, y, P):
    """Mission prediction = argmax of the mean window probabilities."""
    df = pd.DataFrame(P, columns=range(N_CLASSES))
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g.y.first().values, g[list(range(N_CLASSES))].mean().values, g.y.first().index.values


XA_G = torch.from_numpy(XA).to(DEVICE)                    # paired windows on the GPU: (N, 64, 80) and (N, 128, 6)
XS_G = torch.from_numpy(XS).to(DEVICE)
Y_G = torch.from_numpy(YW).to(DEVICE)
Y_HASH = sha16(YW)
WID, WMIS, WDAY, WPHASE = WMETA.window_id.values, WMETA.mission.values, WMETA.day.values, WMETA.phase.values


def fold_prep(k):
    """v29 FOLD_PREP rule: indices, per-feature audio / per-channel IMU z-score from TRAINING windows only, class weights."""
    f = FOLDS[k]
    idx = {s: np.where(WMETA.mission.isin(f[s]).values)[0] for s in ("train", "val", "test")}
    assert set(WMETA.mission.values[idx["train"]]) == set(f["train"])
    a_tr = XA[idx["train"]].reshape(-1, N_AUDIO_FEAT)
    s_tr = XS[idx["train"]].reshape(-1, N_SENSOR_CHANNELS)
    a_mu, a_sd = a_tr.mean(0, dtype=np.float64).astype(np.float32), (a_tr.std(0, dtype=np.float64) + AUDIO_STD_EPS).astype(np.float32)
    s_mu, s_sd = s_tr.mean(0, dtype=np.float64).astype(np.float32), (s_tr.std(0, dtype=np.float64) + SENSOR_STD_EPS).astype(np.float32)
    del a_tr, s_tr
    counts = np.bincount(YW[idx["train"]], minlength=N_CLASSES)
    return {"idx": idx, "a_mu": a_mu, "a_sd": a_sd, "s_mu": s_mu, "s_sd": s_sd, "zhash": sha16(a_mu, a_sd, s_mu, s_sd),
            "class_weights": (counts.sum() / (N_CLASSES * np.maximum(counts, 1))).astype(np.float32)}


FOLD_PREP = {k: fold_prep(k) for k in FOLDS}
hook(11, all(set(WMIS[FOLD_PREP[k]["idx"]["train"]]) == set(FOLDS[k]["train"]) for k in FOLDS),
     "audio and IMU z-score statistics computed from each fold's TRAINING windows only: "
     + ", ".join(f"fold {k} {FOLD_PREP[k]['zhash']}" for k in FOLDS), stop=True)


def zscored(k):
    P = FOLD_PREP[k]
    t = lambda a: torch.from_numpy(a).to(DEVICE)
    return (XA_G - t(P["a_mu"])) / t(P["a_sd"]), (XS_G - t(P["s_mu"])) / t(P["s_sd"])


class Pooled(nn.Module):
    """Wraps an UNCHANGED reference model; forward returns (pooled embedding = the classifier's input, logits)."""

    def __init__(self, model):
        super().__init__()
        self.model = model
        self._z = None
        self.model.classifier.register_forward_pre_hook(self._grab)

    def _grab(self, module, inp):
        self._z = inp[0]

    def forward(self, x):
        logits = self.model(x)
        return self._z, logits


class JointFusion(nn.Module):
    """B2: IMU and audio encoders end-to-end, concatenated pooled embeddings -> linear head (CE only)."""

    def __init__(self):
        super().__init__()
        self.imu = Pooled(IMUMamba())
        self.aud = Pooled(AudioMamba())
        self.head = nn.Linear(2 * D_MODEL, N_CLASSES)

    def forward(self, xa, xs):
        hi, _ = self.imu(xs)
        ha, _ = self.aud(xa)
        return self.head(torch.cat([hi, ha], dim=1))


class ResidualFusion(nn.Module):
    """z_final = z_IMU + alpha * dz; dz = W2 relu(W1 h) with W2, b2 ZERO-initialised; alpha = ALPHA_MAX * sigmoid(g(h))."""

    def __init__(self, anchor, use_audio):
        super().__init__()
        self.anchor = Pooled(anchor)
        self.aud = Pooled(AudioMamba()) if use_audio else None
        d_in = 2 * D_MODEL if use_audio else D_MODEL
        self.w1 = nn.Linear(d_in, R_HIDDEN)
        self.w2 = nn.Linear(R_HIDDEN, N_CLASSES)
        nn.init.zeros_(self.w2.weight)
        nn.init.zeros_(self.w2.bias)
        self.gate = nn.Linear(d_in, 1)
        self.anchor_frozen = True
        self.set_anchor_frozen(True)

    def set_anchor_frozen(self, frozen):
        self.anchor_frozen = frozen
        for p in self.anchor.parameters():
            p.requires_grad_(not frozen)
        if frozen:
            self.anchor.eval()

    def train(self, mode=True):
        super().train(mode)
        if self.anchor_frozen:
            self.anchor.eval()                        # a frozen anchor never updates BatchNorm statistics
        return self

    def forward(self, xa, xs):
        if self.anchor_frozen:
            with torch.no_grad():
                hi, zi = self.anchor(xs)
        else:
            hi, zi = self.anchor(xs)
        h = torch.cat([hi, self.aud(xa)[0]], dim=1) if self.aud is not None else hi
        dz = self.w2(F.relu(self.w1(h)))
        alpha = ALPHA_MAX * torch.sigmoid(self.gate(h))
        return zi + alpha * dz, zi, alpha, dz


def state_hash(module):
    return hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                 for v in module.state_dict().values())).hexdigest()[:16]


def c2_permutation(k, seed):
    """Audio window permutation WITHIN each split (fit / val / test); identity elsewhere. Labels are never touched."""
    rng = np.random.default_rng([C2_PERM_SEED, k, seed])
    perm = np.arange(len(WID))
    shares = {}
    for s, ix in FOLD_PREP[k]["idx"].items():
        perm[ix] = ix[rng.permutation(len(ix))]
        assert set(perm[ix]) == set(ix), f"C2: permutation leaves split {s}"
        shares[s] = float((perm[ix] != ix).mean())
    return perm, shares


def train_loop(tag, model, groups_, loss_fn, eval_fn, fit_idx, seed, epochs, init_best=None, first_batch_check=None):
    """Reference training loop (AdamW, cosine over `epochs`, clip, seeded shuffle, strictly-best val macro-F1)."""
    opt = torch.optim.AdamW(groups_, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=LR_MIN)
    params = [p for g in groups_ for p in g["params"]]
    loader = DataLoader(TensorDataset(torch.from_numpy(fit_idx)), batch_size=BATCH_SIZE, shuffle=True,
                        generator=make_generator(seed))
    best = init_best or {"f1": -1.0, "state": None, "epoch": None}
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        run, nb = 0.0, 0
        for (bi,) in loader:
            bi = bi.to(DEVICE)
            if first_batch_check is not None and ep == 1 and nb == 0:
                first_batch_check(bi)
            opt.zero_grad()
            loss = loss_fn(bi)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=GRAD_CLIP_NORM)
            opt.step()
            run += float(loss)
            nb += 1
        sched.step()
        assert math.isfinite(run), f"{tag} non-finite training loss"
        f1v = eval_fn()
        history.append((round(run / nb, 5), round(f1v, 5)))
        improved = f1v > best["f1"]
        if improved:
            best = {"f1": f1v, "state": copy.deepcopy(model.state_dict()), "epoch": ep}
        print(f"    {tag} epoch {ep:02d}/{epochs} | train loss {run / nb:.4f} | val macro-F1 {f1v:.4f}" + ("  * best" if improved else ""))
    model.load_state_dict(best["state"])
    return best, history


@torch.no_grad()
def predict(fn, idx, bs=256):
    """fn(batch_index_tensor) -> logits or (z_final, z_imu, alpha, dz). Returns probs (+ alpha, ||alpha dz|| if residual)."""
    probs, alphas, norms = [], [], []
    for i in range(0, len(idx), bs):
        bi = torch.from_numpy(idx[i:i + bs]).to(DEVICE)
        out = fn(bi)
        if isinstance(out, tuple):
            zf, _, a, dz = out
            assert bool(((a >= 0) & (a <= ALPHA_MAX)).all()), "alpha outside [0, ALPHA_MAX]"
            ALPHA_CHECKS.append(len(bi))
            alphas.append(a.squeeze(1).float().cpu().numpy())
            norms.append((a * dz).norm(dim=1).float().cpu().numpy())
            out = zf
        probs.append(torch.softmax(out, 1).float().cpu().numpy())
    P = np.concatenate(probs)
    return (P, np.concatenate(alphas), np.concatenate(norms)) if alphas else (P, None, None)


def val_f1(fn, idx):
    P, _, _ = predict(fn, idx)
    return f1_score(YW[idx], P.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0)


ALPHA_CHECKS, ZERO_INIT, ANCHOR_CHECKS, PERM_CHECKS, ID_HASHES = [], [], [], [], {}


def run_arm(arm, k, seed, epochs=EPOCHS, r2_epochs=R2_EPOCHS):
    t0 = time.time()
    set_all_seeds(seed)
    f = FOLDS[k]
    P_ = FOLD_PREP[k]
    fit, va, te = P_["idx"]["train"], P_["idx"]["val"], P_["idx"]["test"]
    # ---- fail-loudly integrity checks ----
    assert f["day"] not in {MISSION_DAY[m] for m in f["train"] + f["val"]}, "test day in fitting/validation"
    assert {MISSION_CLASS[m] for m in f["val"]} == set(range(N_CLASSES)), "inner validation lacks a class"
    assert sha16(YW) == Y_HASH, "labels changed"
    id_hash = hashlib.sha1(("\n".join(WID[te]) + "|" + "\n".join(f["test"]) + "|" +
                            "\n".join(sorted(f["train"] + f["val"]))).encode()).hexdigest()[:16]
    assert ID_HASHES.setdefault((k, seed), id_hash) == id_hash, f"{arm}: test windows / missions / pool differ"
    XA_z, XS_z = zscored(k)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(P_["class_weights"], device=DEVICE))
    tag = f"[{arm} | fold {k} ({f['day']}) | seed {seed}]"
    print(f"{tag} fit {len(f['train'])} missions / {len(fit)} windows | val {len(f['val'])} / {len(va)} | test "
          f"{len(f['test'])} / {len(te)} | z {P_['zhash']}")
    extra = {"b1_w": np.nan, "zero_init_maxdiff": np.nan, "anchor_hash_before": "", "anchor_hash_after": "",
             "anchor_repro_maxdiff": np.nan, "c2_nonidentity_min": np.nan}
    alpha = norm = None
    best = {"epoch": None, "f1": np.nan}
    history = []
    init_sha1 = ""
    if arm in ("B0", "AUD"):
        model = (IMUMamba() if arm == "B0" else AudioMamba()).to(DEVICE)
        assert verify_model(model)[0]
        init_sha1 = state_hash(model)
        fwd = (lambda bi: model(XS_z[bi])) if arm == "B0" else (lambda bi: model(XA_z[bi]))
        best, history = train_loop(tag, model, [{"params": list(model.parameters())}], lambda bi: ce(fwd(bi), Y_G[bi]),
                                   lambda: val_f1(fwd, va), fit, seed, epochs)
        model.eval()
        Pt, _, _ = predict(fwd, te)
        Pv, _, _ = predict(fwd, va)
        torch.save(best["state"], CKPT_DIR / f"{arm}_{k}_{seed}.pt")
    elif arm == "B1":
        zb, za = np.load(PROBS_DIR / f"B0_{k}_{seed}.npz"), np.load(PROBS_DIR / f"AUD_{k}_{seed}.npz")
        assert np.array_equal(zb["val_window_id"], WID[va]) and np.array_equal(za["window_id"], WID[te])
        scores = {w: f1_score(YW[va], (w * zb["probs_val"] + (1 - w) * za["probs_val"]).argmax(1), average="macro",
                              labels=list(range(N_CLASSES)), zero_division=0) for w in B1_WEIGHTS}
        w_best = max(B1_WEIGHTS, key=lambda w: (round(scores[w], 12), w))     # ties -> larger w
        print(f"{tag} validation macro-F1 per w: " + ", ".join(f"{w}: {s:.4f}" for w, s in scores.items()) + f" -> w = {w_best}")
        Pt = w_best * zb["probs"] + (1 - w_best) * za["probs"]
        Pv = w_best * zb["probs_val"] + (1 - w_best) * za["probs_val"]
        extra["b1_w"] = w_best
        best = {"epoch": 0, "f1": scores[w_best]}
    elif arm == "B2":
        model = JointFusion().to(DEVICE)
        init_sha1 = state_hash(model)
        fwd = lambda bi: model(XA_z[bi], XS_z[bi])
        best, history = train_loop(tag, model, [{"params": list(model.parameters())}], lambda bi: ce(fwd(bi), Y_G[bi]),
                                   lambda: val_f1(fwd, va), fit, seed, epochs)
        model.eval()
        Pt, _, _ = predict(fwd, te)
        Pv, _, _ = predict(fwd, va)
    else:                                                  # R1, C1, C2, R2: residual fusion around the B0 anchor
        anchor = IMUMamba().to(DEVICE)
        anchor.load_state_dict(torch.load(CKPT_DIR / f"B0_{k}_{seed}.pt", map_location=DEVICE))
        model = ResidualFusion(anchor, use_audio=(arm != "C1")).to(DEVICE)
        xa_src = XA_z
        if arm == "C2":
            perm, shares = c2_permutation(k, seed)
            xa_src = XA_z[torch.from_numpy(perm).to(DEVICE)]
            extra["c2_nonidentity_min"] = min(shares.values())
            PERM_CHECKS.append((k, seed, shares))
            print(f"{tag} C2 permutation non-identity share per split: {shares}")
        fwd = lambda bi: model(xa_src[bi], XS_z[bi])

        def lossR(bi):
            zf, zi, a, dz = fwd(bi)
            loss = ce(zf, Y_G[bi]) + LAMBDA_R * ((a * dz) ** 2).sum(dim=1).mean()
            if arm == "R2":
                loss = loss + LAMBDA_I * ce(zi, Y_G[bi])
            return loss

        if arm == "R2":
            model.load_state_dict(torch.load(CKPT_DIR / f"R1_{k}_{seed}.pt", map_location=DEVICE))
            model.set_anchor_frozen(False)
            init_sha1 = state_hash(model)
            r1_f1 = val_f1(fwd, va)
            print(f"{tag} incumbent = R1 best state, validation macro-F1 {r1_f1:.4f} (selected_epoch 0 if never beaten)")
            other = [p for n_, p in model.named_parameters() if not n_.startswith("anchor.")]
            best, history = train_loop(tag, model, [{"params": list(model.anchor.parameters()),
                                                     "lr": LEARNING_RATE * R2_IMU_LR_FACTOR},
                                                    {"params": other, "lr": LEARNING_RATE}], lossR,
                                       lambda: val_f1(fwd, va), fit, seed, r2_epochs,
                                       init_best={"f1": r1_f1, "state": copy.deepcopy(model.state_dict()), "epoch": 0})
        else:
            init_sha1 = state_hash(model)
            extra["anchor_hash_before"] = state_hash(model.anchor)
            Pb = np.load(PROBS_DIR / f"B0_{k}_{seed}.npz")["probs"]
            Pa, _, _ = predict(lambda bi: model.anchor(XS_z[bi])[1], te)
            extra["anchor_repro_maxdiff"] = float(np.abs(Pa - Pb).max())
            assert extra["anchor_repro_maxdiff"] < ANCHOR_REPRO_TOL, f"{tag}: anchor does not reproduce B0 ({extra['anchor_repro_maxdiff']:.2e})"

            def zero_check(bi):
                with torch.no_grad():
                    zf, zi, _, _ = fwd(bi)
                d = float((zf - zi).abs().max())
                extra["zero_init_maxdiff"] = d
                ZERO_INIT.append(d)
                assert d < ZERO_INIT_TOL, f"{tag}: z_final != z_IMU at step 0 (max |diff| {d:.2e})"

            trainable = [p for p in model.parameters() if p.requires_grad]
            assert not any(p.requires_grad for p in model.anchor.parameters())
            best, history = train_loop(tag, model, [{"params": trainable}], lossR, lambda: val_f1(fwd, va), fit, seed,
                                       epochs, first_batch_check=zero_check)
            extra["anchor_hash_after"] = state_hash(model.anchor)
            assert extra["anchor_hash_before"] == extra["anchor_hash_after"], f"{tag}: frozen anchor changed"
            ANCHOR_CHECKS.append((arm, k, seed, extra["anchor_repro_maxdiff"]))
        model.eval()
        Pt, alpha, norm = predict(fwd, te)
        Pv, _, _ = predict(fwd, va)
        if arm == "R1":
            torch.save(best["state"], CKPT_DIR / f"R1_{k}_{seed}.pt")
    yt = YW[te]
    pred = Pt.argmax(1)
    my_, mP_, _ = mission_level(WMIS[te], yt, Pt)
    rec = np.array([((pred == c) & (yt == c)).sum() / max((yt == c).sum(), 1) for c in range(N_CLASSES)])
    res = {"arm": arm, "fold": k, "fold_day": f["day"], "seed": seed,
           "window_F1": f1_score(yt, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0),
           "mission_F1": f1_score(my_, mP_.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0),
           "window_accuracy": float((pred == yt).mean()), **{f"recall_c{c}": float(rec[c]) for c in range(N_CLASSES)},
           "faulty_called_normal": float((pred[yt > 0] == 0).mean()), "selected_epoch": best["epoch"],
           "best_val_macro_f1": round(float(best["f1"]), 6), "n_train_missions": len(f["train"]),
           "n_val_missions": len(f["val"]), "n_test_missions": len(f["test"]), "n_test_windows": len(te),
           "id_hash": id_hash, "zstats_sha1": P_["zhash"], "init_sha1": init_sha1, **extra,
           "mean_alpha": float(alpha.mean()) if alpha is not None else np.nan,
           "mean_corr_norm": float(norm.mean()) if norm is not None else np.nan,
           "epochs": r2_epochs if arm == "R2" else (0 if arm == "B1" else epochs), "inner_val_rule": INNER_VAL_RULE,
           "train_seconds": round(time.time() - t0, 1), "history": json.dumps(history)}
    np.savez(PROBS_DIR / f"{arm}_{k}_{seed}.npz", window_id=WID[te].astype(str), mission=WMIS[te].astype(str), cls=yt,
             day=WDAY[te].astype(str), phase=WPHASE[te].astype(str), probs=Pt.astype(np.float32),
             val_window_id=WID[va].astype(str), probs_val=Pv.astype(np.float32),
             alpha=alpha if alpha is not None else np.zeros(0, np.float32),
             corr_norm=norm if norm is not None else np.zeros(0, np.float32))
    if arm not in ("B1",):
        del model
    gc.collect()
    torch.cuda.empty_cache()
    return res

## 5. Pre-registered decision rule (fixed BEFORE the run cell; also written to `e18_rule.txt`)

```text
PRE-REGISTERED DECISION RULE (E18; fixed before any E18 result; applied in code; adapted from E19)
Primary metric: pooled out-of-fold WINDOW macro-F1 from seed-averaged probabilities on the PAIRED test windows
(95 missions, day-out, 5 seeds). Secondary: mission-level macro-F1, per-fold F1, worst-day F1, Class-4 recall,
"faulty called Normal" rate. Every arm is evaluated on exactly the same paired test windows and missions.
Candidates: B1 WEIGHTED, B2 JOINT, R1 RESIDUAL (frozen anchor), R2 RESIDUAL (joint fine-tune) (K = 4),
each against B0 IMU-ONLY retrained on the same paired windows.
An arm is CREDITED only if ALL hold vs B0:
  1. pooled paired delta window macro-F1 >= +0.02;
  2. the 99% paired mission-cluster bootstrap CI of the delta excludes 0 (2000 resamples of missions within class,
     seed 2028, the SAME resampled missions for both arms);
  3. delta > 0 in >= 4 of 5 folds (seed-averaged, per held-out day);
  4. delta > 0 in >= 4 of 5 seeds (each seed's own pooled out-of-fold F1);
  5. worst-day F1 not below B0's (min over folds of the seed-averaged fold F1);
  6. NO-HARM: Class-4 recall not more than 0.05 below B0's, and the "faulty called Normal" rate not more than 0.02
     above B0's (pooled, seed-averaged, window level).
WORSE (mirror): delta <= -0.02, the 99% CI entirely below 0, and delta < 0 in >= 4 of 5 folds.
Otherwise: NOT CREDITED.
Confirmatory contrast (not part of K): R1 and R2 vs B1 with conditions 1-5 -> "BEATS WEIGHTED FUSION" or
"DOES NOT BEAT WEIGHTED FUSION". A residual arm credited vs B0 that does not beat B1 is reported as exactly that.
Control read-out: if C1 (R1-IMUONLY) or C2 (R1-SHUFFLED) reaches conditions 1-3 vs B0, "GAIN NOT ATTRIBUTABLE TO AUDIO"
is printed next to the R1/R2 verdicts. C1, C2 and AUDIO-ONLY are reported, never rated.
Every arm is reported whatever the outcome; a null result is a valid outcome.
```

In [ ]:
# ==============================================================================
# 5b. RUNS — seed-major, then fold, then the arm order B0, AUD, B1, B2, R1, R2, C1, C2 (dependencies first).
#     CSV append after EVERY run; resumable on (arm, fold, seed); runtime estimate BEFORE training.
# ==============================================================================
CSV_FIELDS = ["arm", "fold", "fold_day", "seed", "window_F1", "mission_F1", "window_accuracy"] + \
             [f"recall_c{c}" for c in range(N_CLASSES)] + \
             ["faulty_called_normal", "selected_epoch", "best_val_macro_f1", "b1_w", "mean_alpha", "mean_corr_norm",
              "n_train_missions", "n_val_missions", "n_test_missions", "n_test_windows", "id_hash", "zstats_sha1",
              "init_sha1", "zero_init_maxdiff", "anchor_hash_before", "anchor_hash_after", "anchor_repro_maxdiff",
              "c2_nonidentity_min", "epochs", "inner_val_rule", "train_seconds", "history", "finished_at"]


def load_done():
    if not RESULTS_CSV.exists():
        return {}
    df = pd.read_csv(RESULTS_CSV)
    out = {}
    for r in df.itertuples():
        key = (r.arm, int(r.fold), int(r.seed))
        ok = (PROBS_DIR / f"{r.arm}_{int(r.fold)}_{int(r.seed)}.npz").exists()
        if r.arm in CKPT_ARMS:
            ok &= (CKPT_DIR / f"{r.arm}_{int(r.fold)}_{int(r.seed)}.pt").exists()
        if ok:
            out[key] = r
    return out


def append_row(res):
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**res, "finished_at": datetime.now().isoformat(timespec="seconds")})


PLAN = [(a, k, s) for s in SEEDS_TO_RUN for k in FOLDS_TO_RUN for a in ARMS_TO_RUN]
_done = load_done()
_todo = [r for r in PLAN if r not in _done]
_scale = lambda a: (R2_EPOCHS_TO_RUN / R2_EPOCHS) if a == "R2" else (EPOCHS_TO_RUN / EPOCHS)
_est = sum(SEC_PER_RUN[a] * _scale(a) for a, _, _ in _todo)
print(f"{len(PLAN)} runs planned ({len(ARMS_TO_RUN)} arms x {len(FOLDS_TO_RUN)} folds x {len(SEEDS_TO_RUN)} seeds) | "
      f"already done: {len(PLAN) - len(_todo)} | to train now: {len(_todo)}")
print(f"ESTIMATED training time: {_est / 3600:.2f} h (+ parsing on a first session) — budget {BUDGET_HOURS} h")
if _est / 3600 > BUDGET_HOURS:
    loud(f"The estimate {_est / 3600:.2f} h exceeds the {BUDGET_HOURS} h budget. Nothing is reduced automatically:\n"
         "set RUN_SUBSET (top cell) to run part of the plan now; finished runs are skipped when you resume.")
print("Note: mamba-ssm's CUDA kernels are not bit-deterministic, so repeated B0 runs can differ slightly; every arm of a "
      "(fold, seed) uses the SAME B0 checkpoint, so the residual comparisons are unaffected by this noise.")
_t0, _n = time.time(), 0
for (arm, k, s) in PLAN:
    if (arm, k, s) in _done:
        print(f"[{arm} fold {k} seed {s}] already in {RESULTS_CSV.name} — skipped")
        continue
    for dep in DEPENDS.get(arm, []):
        assert (dep, k, s) in _done or (dep, k, s) in load_done(), f"{arm} fold {k} seed {s}: dependency {dep} missing"
    res = run_arm(arm, k, s, epochs=EPOCHS_TO_RUN, r2_epochs=R2_EPOCHS_TO_RUN)
    append_row(res)
    _done[(arm, k, s)] = res
    _n += 1
    _el = time.time() - _t0
    print(f"[{_n}/{len(_todo)}] {arm} fold {k} ({res['fold_day']}) seed {s} | window F1 {res['window_F1']:.4f} | mission F1 "
          f"{res['mission_F1']:.4f} | epoch {res['selected_epoch']}" + (f" | w {res['b1_w']}" if arm == "B1" else "")
          + f" | {res['train_seconds']:.0f} s | ETA {(len(_todo) - _n) * _el / _n / 60:.0f} min")
print(f"\nTraining finished in {(time.time() - _t0) / 60:.1f} min | runs in CSV: {len(load_done())}/{len(PLAN)}")

# ---- run-level hooks (asserted per run; summarised here) ----
_df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(columns=CSV_FIELDS)
_same = _df.groupby(["fold", "seed"]).id_hash.nunique().max() if len(_df) else 1
hook(5, _same == 1, f"test window ids + test missions + training pool identical across arms in every (fold, seed): max "
                    f"distinct hashes {_same}", stop=True)
_res_rows = _df[_df.arm.isin(["R1", "C1", "C2"])]
hook(7, bool(len(_res_rows) == 0 or (_res_rows.zero_init_maxdiff < ZERO_INIT_TOL).all()),
     f"{len(_res_rows)} residual runs: max |z_final - z_IMU| at step 0 = {_res_rows.zero_init_maxdiff.max() if len(_res_rows) else 'n/a'}",
     stop=True)
hook(8, bool(len(_res_rows) == 0 or ((_res_rows.anchor_hash_before == _res_rows.anchor_hash_after).all()
                                     and (_res_rows.anchor_repro_maxdiff < ANCHOR_REPRO_TOL).all())),
     f"frozen anchor unchanged in all {len(_res_rows)} R1/C1/C2 runs; anchor reproduces B0 test probabilities (max |diff| "
     f"{_res_rows.anchor_repro_maxdiff.max() if len(_res_rows) else 'n/a'})", stop=True)
hook(9, True, f"alpha within [0, {ALPHA_MAX}] asserted on every evaluation pass ({sum(ALPHA_CHECKS):,} windows checked this "
              "session)", stop=True)
_c2 = _df[_df.arm == "C2"]
hook(10, bool(len(_c2) == 0 or (_c2.c2_nonidentity_min > 0.95).all()),
     f"C2: permutation stays inside train / val / test (asserted), min non-identity share over splits and runs "
     f"{_c2.c2_nonidentity_min.min() if len(_c2) else 'n/a'}; labels hash unchanged ({Y_HASH})", stop=True)

## 6. Results (all arms on the same paired test windows; 5 seeds)
- **Verdicts** are applied in code from the pre-registered rule: K = 4 candidates vs B0, plus the confirmatory
  contrast vs B1 and the control read-out.
- **Residual diagnostics** for R1, R2, C1 and C2: mean α; mean ‖αΔz‖; the share of test windows whose argmax differs
  from the B0 anchor; windows corrected (anchor wrong → fusion right) and broken (anchor right → fusion wrong); net.
  These are split by true class and by whether the anchor's error was adjacent-class.

In [ ]:
# ==============================================================================
# 6a. RESULT HELPERS — audio reference functions (verbatim) + E18 pooling, rule, diagnostics
# ==============================================================================
def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Resample MISSIONS within class (never windows). Returns bootstrap confusion matrices (B, 5, 5)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    cnt = np.zeros((len(ms), N_CLASSES), int)
    pos = {m: i for i, m in enumerate(ms)}
    mcls = np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        draw = rows[rng.integers(0, len(rows), (B, len(rows)))]
        cms[:, c, :] = cnt[draw].sum(1)
    return cms


def boot_summary(missions, y, pred):
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    p, r, f = prf_from_cm(cm)
    cms = cluster_bootstrap(missions, y, pred)
    bp, br, bf = zip(*(prf_from_cm(c) for c in cms))
    bp, br, bf = np.array(bp), np.array(br), np.array(bf)
    ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
    tab = pd.DataFrame({"precision": p, "P CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bp).T],
                        "recall": r, "R CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(br).T],
                        "f1": f, "F1 CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bf).T]},
                       index=[f"class {c}" for c in range(N_CLASSES)])
    macro = (f.mean(), *ci(bf.mean(1)))
    bal = (r.mean(), *ci(br.mean(1)))
    return tab, macro, bal, cm, (r, ci(br))


def macro_f1(yt, pred):
    return float(f1_score(yt, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0))


def pool_arm(arm):
    per_seed, ids0, meta = {}, None, None
    for s in SEEDS:
        parts = [dict(np.load(PROBS_DIR / f"{arm}_{k}_{s}.npz")) for k in range(len(DAYS))]
        ids = np.concatenate([p["window_id"] for p in parts])
        assert len(ids) == len(set(ids)) == len(WID), f"{arm} seed {s}: pooled windows"
        order = np.argsort(ids)
        per_seed[s] = np.concatenate([p["probs"] for p in parts])[order]
        if ids0 is None:
            ids0 = ids[order]
            meta = pd.DataFrame({c: np.concatenate([p[c] for p in parts])[order] for c in ("mission", "cls", "day")})
        assert np.array_equal(ids[order], ids0)
    return ids0, meta, per_seed


def arm_stats(meta, per_seed):
    yt = meta.cls.values.astype(int)
    P = np.mean(list(per_seed.values()), axis=0)
    pred = P.argmax(1)
    _, macro, bal, cm, (rec, _) = boot_summary(meta.mission.values, yt, pred)
    my_, mP_, mn_ = mission_level(meta.mission.values, yt, P)
    _, mmacro, _, _, _ = boot_summary(mn_, my_, mP_.argmax(1))
    fold_f1 = {d: macro_f1(yt[(meta.day == d).values], pred[(meta.day == d).values]) for d in DAYS}
    return {"window_f1": macro, "mission_f1": mmacro, "recall": rec, "fold_f1": fold_f1, "min_fold": min(fold_f1.values()),
            "worst_day": min(fold_f1, key=fold_f1.get), "seed_f1": {s: macro_f1(yt, v.argmax(1)) for s, v in per_seed.items()},
            "c4_recall": float(rec[4]), "fcn": float((pred[yt > 0] == 0).mean()), "pred": pred, "y": yt,
            "boot": np.array([prf_from_cm(c)[2].mean() for c in cluster_bootstrap(meta.mission.values, yt, pred)])}


def compare(st, ref):
    """Paired comparison of an arm with a reference arm under the pre-registered conditions."""
    delta = st["window_f1"][0] - ref["window_f1"][0]
    q = (100 - PAIRED_CI) / 2
    lo, hi = np.percentile(st["boot"] - ref["boot"], [q, 100 - q])
    fd = {d: st["fold_f1"][d] - ref["fold_f1"][d] for d in DAYS}
    sd_ = {s: st["seed_f1"][s] - ref["seed_f1"][s] for s in ref["seed_f1"]}
    c = {"c1_delta": delta >= MIN_DELTA, "c2_ci": lo > 0, "c3_folds": sum(v > 0 for v in fd.values()) >= MIN_FOLDS_SAME_SIGN,
         "c4_seeds": sum(v > 0 for v in sd_.values()) >= MIN_SEEDS_POSITIVE, "c5_worst_day": st["min_fold"] >= ref["min_fold"],
         "c6_no_harm": (st["c4_recall"] >= ref["c4_recall"] - NO_HARM_C4_RECALL) and (st["fcn"] <= ref["fcn"] + NO_HARM_FCN)}
    worse = delta <= -MIN_DELTA and hi < 0 and sum(v < 0 for v in fd.values()) >= MIN_FOLDS_SAME_SIGN
    return {"delta": float(delta), "ci_lo": float(lo), "ci_hi": float(hi), "folds_pos": sum(v > 0 for v in fd.values()),
            "seeds_pos": sum(v > 0 for v in sd_.values()), "min_fold": st["min_fold"], "min_fold_ref": ref["min_fold"],
            "c4_recall": st["c4_recall"], "c4_recall_ref": ref["c4_recall"], "fcn": st["fcn"], "fcn_ref": ref["fcn"],
            "mission_delta": float(st["mission_f1"][0] - ref["mission_f1"][0]), **c, "worse": worse,
            "fold_delta": fd, "seed_delta": sd_}


def verdict_vs_b0(c):
    if all(c[k] for k in ("c1_delta", "c2_ci", "c3_folds", "c4_seeds", "c5_worst_day", "c6_no_harm")):
        return "CREDITED"
    return "WORSE" if c["worse"] else "NOT CREDITED"


def verdict_vs_b1(c):
    return "BEATS WEIGHTED FUSION" if all(c[k] for k in ("c1_delta", "c2_ci", "c3_folds", "c4_seeds", "c5_worst_day")) \
        else "DOES NOT BEAT WEIGHTED FUSION"


def residual_diagnostics(arm, k, s):
    z = np.load(PROBS_DIR / f"{arm}_{k}_{s}.npz")
    zb = np.load(PROBS_DIR / f"B0_{k}_{s}.npz")
    assert np.array_equal(z["window_id"], zb["window_id"])
    yt, pf, pa = z["cls"].astype(int), z["probs"].argmax(1), zb["probs"].argmax(1)
    corr, brok = (pa != yt) & (pf == yt), (pa == yt) & (pf != yt)
    adj = np.abs(pa - yt) == 1
    row = {"arm": arm, "fold": k, "fold_day": DAYS[k], "seed": s, "mean_alpha": float(z["alpha"].mean()),
           "mean_alpha_dz_norm": float(z["corr_norm"].mean()), "share_argmax_differs": float((pf != pa).mean()),
           "corrected": int(corr.sum()), "broken": int(brok.sum()), "net": int(corr.sum() - brok.sum()),
           "corrected_anchor_adjacent_err": int((corr & adj).sum()), "corrected_anchor_distant_err": int((corr & ~adj).sum()),
           "broken_to_adjacent_err": int((brok & (np.abs(pf - yt) == 1)).sum()),
           "broken_to_distant_err": int((brok & (np.abs(pf - yt) >= 2)).sum())}
    for c in range(N_CLASSES):
        row[f"corrected_c{c}"] = int((corr & (yt == c)).sum())
        row[f"broken_c{c}"] = int((brok & (yt == c)).sum())
    return row

In [ ]:
# ==============================================================================
# 6b. POOLED RESULTS, PAIRED TABLES (vs B0 and confirmatory vs B1), PER-FOLD, RESIDUAL DIAGNOSTICS, SOFT HOOK 12
# ==============================================================================
_done = load_done()
COMPLETE = [a for a in ARMS if all((a, k, s) in _done for k in range(len(DAYS)) for s in SEEDS)]
STATS, CMP, CMP_B1, CONTROL_FLAG = {}, {}, {}, False
TABLE1 = TABLE2 = PERFOLD = DIAG = None
if SMOKE:
    print("SMOKE run: pooled analysis skipped (it needs 5 folds x 5 seeds). Smoke rows:")
    print(_df[["arm", "fold", "seed", "window_F1", "mission_F1", "selected_epoch", "b1_w", "mean_alpha",
               "zero_init_maxdiff", "anchor_repro_maxdiff", "train_seconds"]].to_string(index=False))
elif "B0" not in COMPLETE:
    loud(f"B0 is incomplete (complete arms: {COMPLETE}); the paired analysis needs B0 for all 5 folds x 5 seeds.")
else:
    ids_ref = None
    for arm in COMPLETE:
        ids, meta, per_seed = pool_arm(arm)
        ids_ref = ids if ids_ref is None else ids_ref
        assert np.array_equal(ids, ids_ref), f"{arm}: different pooled windows — not paired"
        STATS[arm] = arm_stats(meta, per_seed)
    for arm in [a for a in CANDIDATES + CONTROLS + ["AUD"] if a in COMPLETE]:
        CMP[arm] = compare(STATS[arm], STATS["B0"])
    for arm in [a for a in ("R1", "R2") if a in COMPLETE and "B1" in COMPLETE]:
        CMP_B1[arm] = compare(STATS[arm], STATS["B1"])
    CONTROL_FLAG = any(CMP[c]["c1_delta"] and CMP[c]["c2_ci"] and CMP[c]["c3_folds"] for c in CONTROLS if c in CMP)

    _t1 = []
    for arm in COMPLETE:
        st = STATS[arm]
        rr = _df[_df.arm == arm]
        _t1.append({"arm": arm, "name": ARM_NAMES[arm],
                    "window F1 [95% CI]": f"{st['window_f1'][0]:.4f} [{st['window_f1'][1]:.4f}, {st['window_f1'][2]:.4f}]",
                    "mission F1 [95% CI]": f"{st['mission_f1'][0]:.4f} [{st['mission_f1'][1]:.4f}, {st['mission_f1'][2]:.4f}]",
                    **{f"seed {s} F1": round(v, 4) for s, v in st["seed_f1"].items()},
                    "seed SD": round(float(np.std(list(st["seed_f1"].values()), ddof=1)), 4),
                    "worst day": f"{st['worst_day']} ({st['min_fold']:.4f})",
                    **{f"F1 {d}": round(v, 4) for d, v in st["fold_f1"].items()},
                    **{f"recall c{c}": round(float(st["recall"][c]), 3) for c in range(N_CLASSES)},
                    "faulty called Normal": round(st["fcn"], 4),
                    "mean selected epoch": round(float(rr.selected_epoch.mean()), 1),
                    "B1 w (mean)": round(float(rr.b1_w.mean()), 2) if arm == "B1" else ""})
    TABLE1 = pd.DataFrame(_t1)

    def _row(arm, c, vs, verdict):
        return {"arm": arm, "name": ARM_NAMES[arm], "vs": vs, "delta": round(c["delta"], 4), "CI_lo_99": round(c["ci_lo"], 4),
                "CI_hi_99": round(c["ci_hi"], 4), "folds_positive": c["folds_pos"], "seeds_positive": c["seeds_pos"],
                "worst_day_F1": round(c["min_fold"], 4), f"worst_day_F1_{vs}": round(c["min_fold_ref"], 4),
                "class4_recall": round(c["c4_recall"], 4), f"class4_recall_{vs}": round(c["c4_recall_ref"], 4),
                "faulty_called_normal": round(c["fcn"], 4), f"faulty_called_normal_{vs}": round(c["fcn_ref"], 4),
                "delta_mission_F1": round(c["mission_delta"], 4),
                **{k_: bool(c[k_]) for k_ in ("c1_delta", "c2_ci", "c3_folds", "c4_seeds", "c5_worst_day", "c6_no_harm")},
                "verdict": verdict}

    _t2 = []
    for arm in [a for a in CANDIDATES if a in CMP]:
        v = verdict_vs_b0(CMP[arm])
        if arm in ("R1", "R2"):
            if arm in CMP_B1:
                vb1 = verdict_vs_b1(CMP_B1[arm])
                if v == "CREDITED" and vb1 != "BEATS WEIGHTED FUSION":
                    v = "CREDITED vs IMU-only, but DOES NOT BEAT WEIGHTED FUSION"
            if CONTROL_FLAG:
                v += " | GAIN NOT ATTRIBUTABLE TO AUDIO"
        _t2.append(_row(arm, CMP[arm], "B0", v))
    for arm in [a for a in CONTROLS + ["AUD"] if a in CMP]:
        _t2.append(_row(arm, CMP[arm], "B0", "control / reference (not rated)"))
    for arm, c in CMP_B1.items():
        _t2.append(_row(arm, c, "B1", verdict_vs_b1(c)))
    for arm in [a for a in CANDIDATES if a not in CMP]:
        _t2.append({"arm": arm, "name": ARM_NAMES[arm], "vs": "B0", "verdict": "NOT RUN / INCOMPLETE"})
    TABLE2 = pd.DataFrame(_t2)
    PERFOLD = pd.DataFrame([{"arm": a, "fold_day": d, "window_F1_seedavg": STATS[a]["fold_f1"][d],
                             "delta_vs_B0": STATS[a]["fold_f1"][d] - STATS["B0"]["fold_f1"][d]} for a in COMPLETE for d in DAYS])
    DIAG = pd.DataFrame([residual_diagnostics(a, k, s) for a in ("R1", "R2", "C1", "C2") if a in COMPLETE
                         for k in range(len(DAYS)) for s in SEEDS])
    TABLE1.to_csv(OUT / "e18_table_results.csv", index=False)
    TABLE2.to_csv(OUT / "e18_table_paired.csv", index=False)
    PERFOLD.to_csv(OUT / "e18_perfold.csv", index=False)
    DIAG.to_csv(OUT / "e18_residual_diagnostics.csv", index=False)
    pd.set_option("display.width", 250)
    print("=== Pooled out-of-fold results on the paired windows (seed-averaged, 5 seeds) ===")
    print(TABLE1.set_index("arm").T.to_string())
    print(f"\n=== Paired comparisons ({PAIRED_CI:.0f}% CI) and VERDICTS (vs B0: K = {len(CANDIDATES)}; vs B1: confirmatory) ===")
    print(TABLE2.set_index(["arm", "vs"])[["delta", "CI_lo_99", "CI_hi_99", "folds_positive", "seeds_positive",
                                           "c5_worst_day", "c6_no_harm", "verdict"]].to_string())
    if CONTROL_FLAG:
        print("\nCONTROL READ-OUT: a control (C1 and/or C2) reaches conditions 1-3 vs B0 -> GAIN NOT ATTRIBUTABLE TO AUDIO")
    if len(DIAG):
        print("\n=== Residual diagnostics (sums over folds x seeds) ===")
        print(DIAG.groupby("arm")[["mean_alpha", "mean_alpha_dz_norm", "share_argmax_differs"]].mean().round(4)
              .join(DIAG.groupby("arm")[["corrected", "broken", "net"]].sum()).to_string())

    # ---- hook 12 (SOFT): B0 on paired windows vs E19 A0 (IMU) per fold ----
    _e19 = [p for p in [OUT / "e19_results.csv"] + [Path(x) for x in sorted(glob.glob("/kaggle/input/**/e19_results.csv", recursive=True))] if p.exists()]
    if _e19:
        _a0 = pd.read_csv(_e19[0])
        _a0 = _a0[(_a0.modality == "imu") & (_a0.arm == "A0")].groupby("fold").window_F1.mean().reindex(range(len(DAYS)))
        _ref, _src = _a0.values, f"E19 A0 ({_e19[0]})"
    else:
        _ref, _src = np.array(E19_A0_FALLBACK), "IMU reference per-fold F1 (no e19_results.csv found)"
    _b0 = _df[_df.arm == "B0"].groupby("fold").window_F1.mean().reindex(range(len(DAYS))).values
    _dev = _b0 - _ref
    hook(12, bool(np.all(np.abs(_dev) <= B0_REF_TOL)),
         f"B0 (paired windows) vs {_src}: " + ", ".join(f"{DAYS[i]} {_b0[i]:.4f} vs {_ref[i]:.4f} ({_dev[i]:+.4f})"
                                                      for i in range(len(DAYS)))
         + ("" if np.all(np.abs(_dev) <= B0_REF_TOL) else
            f" — WARN: outside ±{B0_REF_TOL} (different windows: paired windows start at audio onset; mamba-ssm CUDA "
            "non-determinism)"), stop=False)

In [ ]:
# ==============================================================================
# 7. SUMMARY (generated from the numbers; verdicts come from the rule in code) -> e18_summary.txt; zip; hooks
# ==============================================================================
import zipfile

_L = ["WHAT E18 TELLS US", ""]
if SMOKE:
    _L += ["SMOKE run only (fold 1, seed 42, 2 epochs, every arm): checks that the pipeline runs end to end. No scientific "
           "conclusion can be drawn from it."]
elif "B0" not in STATS:
    _L += ["B0 incomplete: no analysis yet. Re-run the training cell (it resumes)."]
else:
    _L.append(f"Pairing: v29 paired_windows (v24 CORRECTED_ALIGNMENT); {len(WID)} paired windows, {100 * REAL_AUDIO_SHARE:.1f}% "
              f"with real audio, {100 * IMU_GRID_COVERAGE:.1f}% of the IMU-only reference grid covered. Inner validation: "
              f"{INNER_VAL_RULE}.")
    _L.append("")
    for arm in COMPLETE:
        st = STATS[arm]
        line = (f"{arm} {ARM_NAMES[arm]}: window F1 {st['window_f1'][0]:.4f} [{st['window_f1'][1]:.4f}, {st['window_f1'][2]:.4f}], "
                f"mission F1 {st['mission_f1'][0]:.4f}, worst day {st['worst_day']} ({st['min_fold']:.4f}), Class-4 recall "
                f"{st['c4_recall']:.3f}, faulty called Normal {st['fcn']:.4f}")
        if arm == "B1":
            line += f", w chosen on validation: mean {_df[_df.arm == 'B1'].b1_w.mean():.2f} ({sorted(_df[_df.arm == 'B1'].b1_w.unique().tolist())})"
        _L.append(line)
    _L += ["", "Verdicts (pre-registered rule, applied in code):"]
    for r in TABLE2.itertuples():
        if r.vs == "B0" and r.arm in CANDIDATES:
            _L.append(f"- {r.arm} vs B0: " + (f"delta {r.delta:+.4f} [{r.CI_lo_99:+.4f}, {r.CI_hi_99:+.4f}], folds+ "
                                              f"{r.folds_positive}/5, seeds+ {r.seeds_positive}/5 -> " if pd.notna(getattr(r, 'delta', np.nan)) else "")
                      + f"{r.verdict}")
    for r in TABLE2.itertuples():
        if r.vs == "B1":
            _L.append(f"- {r.arm} vs B1 (confirmatory): delta {r.delta:+.4f} [{r.CI_lo_99:+.4f}, {r.CI_hi_99:+.4f}] -> {r.verdict}")
    for r in TABLE2.itertuples():
        if r.arm in CONTROLS + ["AUD"]:
            _L.append(f"- {r.arm} ({ARM_NAMES[r.arm]}) vs B0: delta {r.delta:+.4f} [{r.CI_lo_99:+.4f}, {r.CI_hi_99:+.4f}] "
                      f"(reported, not rated)")
    _L.append("Control read-out: " + ("GAIN NOT ATTRIBUTABLE TO AUDIO (a control reaches conditions 1-3 vs B0)."
                                      if CONTROL_FLAG else "no control reaches conditions 1-3 vs B0."))
    if DIAG is not None and len(DIAG):
        _g = DIAG.groupby("arm")
        _L.append("Residual corrections (sum over 5 folds x 5 seeds): " + "; ".join(
            f"{a}: corrected {int(g.corrected.sum())}, broken {int(g.broken.sum())}, net {int(g.net.sum()):+d}, mean alpha "
            f"{g.mean_alpha.mean():.3f}, mean ||alpha dz|| {g.mean_alpha_dz_norm.mean():.3f}" for a, g in _g))
_L += ["",
       "Limits",
       "- 95 missions (~19 per class) over 5 recording days; one day (2021-12-22) has only 10 missions; 5 seeds.",
       f"- K = {len(CANDIDATES)} candidate arms (99% paired CIs); fixed LAMBDA_R = {LAMBDA_R}, LAMBDA_I = {LAMBDA_I}, "
       f"ALPHA_MAX = {ALPHA_MAX}, R_HIDDEN = {R_HIDDEN}, never tuned.",
       "- Pairing: v29 paired_windows (v24 CORRECTED_ALIGNMENT: audio anchored at recording_started_time, chunk duration "
       "from the mission length); windows start at audio onset, so the IMU-only stretch before audio is excluded from every arm.",
       "- mamba-ssm CUDA kernels are not bit-deterministic; B0 vs E19 A0 is only a soft check.",
       "- Checkpoint selection on validation means R1 rarely falls far below its anchor (it starts AT the anchor), so "
       "'no harm' alone is not a result; only the credit conditions vs B0 and the contrast vs B1 are."]
SUMMARY = "\n".join(_L)
print(SUMMARY)
(OUT / ("e18_summary_smoke.txt" if SMOKE else "e18_summary.txt")).write_text(SUMMARY, encoding="utf-8")

_zip = OUT / ("e18_outputs_smoke.zip" if SMOKE else "e18_outputs.zip")
_n = 0
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("e18_*.csv")) + sorted(OUT.glob("e18_*.txt")) + sorted(PROBS_DIR.glob("*.npz")):
        zf.write(p, p.relative_to(OUT))
        _n += 1
print(f"\nWrote {_zip} ({_n} files, {_zip.stat().st_size / 1e6:.1f} MB; caches, checkpoints and the dataset excluded)")

print("\n" + "=" * 110)
print("VERIFICATION HOOKS (hook 12 is SOFT: WARN only)")
print("=" * 110)
for n in sorted(HOOK_NAMES):
    res = HOOKS.get(n, [])
    status = "NOT RUN" if not res else ("PASS" if all(ok for ok, _ in res) else ("WARN" if n == 12 else "FAIL"))
    print(f"  HOOK {n:<2} {status:<8} {HOOK_NAMES[n]}")
    for ok, msg in res:
        print(f"      {'PASS' if ok else ('WARN' if n == 12 else 'FAIL')}: {msg}")
print("=" * 110)
print("\n💾 Click \"Save Version\" -> \"Save & Run All (Commit)\" to keep /kaggle/working (or download e18_outputs.zip).")